# Smart Tomato Care 🍅

> Growing Smarter, Harvesting Better.



## 📦 Import Libraries & Install Dependencies


>





In [ ]:
import re
import json
import io
import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import google.generativeai as genai
from collections import defaultdict
from nltk.stem import PorterStemmer
from datetime import datetime
from PIL import Image
from transformers import pipeline
from IPython.display import HTML, display
from ipywidgets import Layout
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

!pip install -q requests nltk scikit-learn ipywidgets

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.1 MB/s eta 0:00:00


## 🔍 Search Engine

In [ ]:
class FirebaseRAGIndexService:
    """
    Search engine + simple RAG system.

    It reads articles_json from Firebase Realtime Database.

    Expected Firebase path:
        https://smart-tomato-care-default-rtdb.firebaseio.com/articles_json.json

    Expected Firebase structure:
        {
            "1": {
                "DocID": "1",
                "title": "...",
                "file": "...",
                "url": "...",
                "text": "full extracted article text..."
            }
        }

    What this service does:
    1. Reads articles from Firebase JSON
    2. Checks if index already exists in Firebase (singleton pattern)
    3. If yes: loads index directly from Firebase (fast)
    4. If no: builds the 20-term inverted index and saves it to Firebase
    5. Splits articles into smaller chunks
    6. Builds a TF-IDF vector store
    7. Retrieves the most relevant chunks
    8. Generates a RAG-style answer before showing document links
    """

    def __init__(
        self,
        firebase_url,
        articles_path="articles_json",
        chunk_size=900,
        chunk_overlap=150,
        min_similarity=0.04
    ):
        self.firebase_url = firebase_url.rstrip("/")
        self.articles_path = articles_path

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.min_similarity = min_similarity

        self.stemmer = PorterStemmer()

        self.significant_terms = {
            "tomato",
            "heirloom",
            "soil",
            "plants",
            "food",
            "systems",
            "production",
            "fruit",
            "farmers",
            "organic",
            "varieties",
            "number",
            "agriculture",
            "sustainable",
            "growth",
            "local",
            "quality",
            "growing",
            "crop",
            "agricultural"
        }

        self.term_stems = {
            self.stemmer.stem(term): term
            for term in self.significant_terms
        }

        self.stop_words = {
            "the", "and", "of", "in", "to", "a", "for", "as", "with",
            "that", "is", "on", "are", "by", "was", "were", "from",
            "or", "this", "at", "an", "be", "it", "its", "into",
            "their", "they", "these", "those", "which", "also",
            "has", "have", "had", "can", "could", "may", "more",
            "than", "then", "there", "such", "using", "used"
        }

        self.index = defaultdict(lambda: {
            "count": 0,
            "DocIDs": set(),
            "entries": [],
            "seen_titles": set(),
            "doc_counts": defaultdict(int)
        })

        self.documents = {}
        self.articles_json = {}

        self.chunks = []
        self.vectorizer = None
        self.chunk_matrix = None

    def fetch_articles_from_firebase(self):
        """
        Downloads articles_json from Firebase Realtime Database.
        """

        endpoint = f"{self.firebase_url}/{self.articles_path}.json"

        print("⬇️ Reading articles from Firebase:")
        print(endpoint)

        response = requests.get(endpoint)

        if response.status_code != 200:
            raise Exception(
                f"Firebase read failed.\n"
                f"Status code: {response.status_code}\n"
                f"Response: {response.text[:500]}"
            )

        data = response.json()

        if data is None:
            raise Exception(
                "No data found in Firebase. Make sure articles_json exists."
            )

        if isinstance(data, list):
            fixed_data = {}

            for item in data:
                if item and isinstance(item, dict):
                    doc_id = str(item.get("DocID", len(fixed_data) + 1))
                    fixed_data[doc_id] = item

            data = fixed_data

        self.articles_json = data

        print(f"✅ Loaded {len(self.articles_json)} articles from Firebase.")

    def clean_and_tokenize(self, text):
        """
        Converts text to lowercase and extracts clean English words.
        """

        words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

        clean_words = []

        for word in words:
            if word not in self.stop_words and len(word) > 2:
                clean_words.append(word)

        return clean_words

    def normalize_word_to_term(self, word):
        """
        Converts a word into one of the selected 20 terms using stemming.
        """

        stemmed_word = self.stemmer.stem(word.lower())

        if stemmed_word in self.term_stems:
            return self.term_stems[stemmed_word]

        return None

    def build_index_for_article(self, doc_id, article):
        """
        Builds the inverted index from one article.
        """

        title = article.get("title", "No Title")
        file_name = article.get("file", "Unknown File")
        url = article.get("url", "#")
        text = article.get("text", "")

        if not text.strip():
            print(f"⚠️ No text found for DocID {doc_id}: {title}")
            return

        self.documents[doc_id] = {
            "DocID": doc_id,
            "title": title,
            "file": file_name,
            "url": url
        }

        words = self.clean_and_tokenize(text)

        for word in words:
            original_term = self.normalize_word_to_term(word)

            if original_term is not None:
                self.index[original_term]["count"] += 1
                self.index[original_term]["DocIDs"].add(doc_id)
                self.index[original_term]["doc_counts"][doc_id] += 1

                key_title = title.strip().lower()

                if key_title not in self.index[original_term]["seen_titles"]:
                    self.index[original_term]["entries"].append({
                        "DocID": doc_id,
                        "title": title,
                        "file": file_name,
                        "url": url
                    })

                    self.index[original_term]["seen_titles"].add(key_title)

    def split_text_into_chunks(self, text):
        """
        Splits long article text into smaller overlapping chunks.
        This makes retrieval more accurate than searching the whole article.
        """

        words = text.split()

        if not words:
            return []

        chunks = []
        start = 0

        while start < len(words):
            end = start + self.chunk_size
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            if chunk_text.strip():
                chunks.append(chunk_text)

            start += self.chunk_size - self.chunk_overlap

        return chunks

    def build_rag_chunks(self):
        """
        Creates searchable text chunks from all articles.
        """

        self.chunks = []

        for doc_id, article in self.articles_json.items():
            doc_id = str(doc_id)

            title = article.get("title", "No Title")
            file_name = article.get("file", "Unknown File")
            url = article.get("url", "#")
            text = article.get("text", "")

            article_chunks = self.split_text_into_chunks(text)

            for chunk_index, chunk_text in enumerate(article_chunks, start=1):
                self.chunks.append({
                    "chunk_id": f"{doc_id}_{chunk_index}",
                    "DocID": doc_id,
                    "title": title,
                    "file": file_name,
                    "url": url,
                    "text": chunk_text
                })

        print(f"✅ Created {len(self.chunks)} text chunks for RAG.")

    def build_vector_store(self):
        """
        Builds a simple TF-IDF vector store.
        """

        if not self.chunks:
            raise Exception("No chunks found. Cannot build vector store.")

        chunk_texts = [chunk["text"] for chunk in self.chunks]

        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=8000,
            ngram_range=(1, 2)
        )

        self.chunk_matrix = self.vectorizer.fit_transform(chunk_texts)

        print("✅ TF-IDF vector store is ready.")

    def retrieve_context(self, query, top_k=4):
        """
        Retrieves the most relevant chunks using cosine similarity.
        """

        if self.vectorizer is None or self.chunk_matrix is None:
            raise Exception("Vector store is not built yet.")

        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.chunk_matrix)[0]

        ranked_indices = similarities.argsort()[::-1]

        retrieved = []

        for index in ranked_indices[:top_k]:
            score = float(similarities[index])

            if score < self.min_similarity:
                continue

            chunk = self.chunks[index].copy()
            chunk["similarity"] = score
            retrieved.append(chunk)

        return retrieved

    def sentence_split(self, text):
        """
        Splits text into readable sentences.
        """

        sentences = re.split(r"(?<=[.!?])\s+", text)

        clean_sentences = []

        for sentence in sentences:
            sentence = sentence.strip()

            if len(sentence) >= 60:
                clean_sentences.append(sentence)

        return clean_sentences

    def get_query_keywords(self, query):
        """
        Extracts important query words.
        """

        words = re.findall(r"\b[a-zA-Z]+\b", query.lower())

        keywords = []

        for word in words:
            if word not in self.stop_words and len(word) > 2:
                keywords.append(word)

        return keywords

    def generate_rag_answer(self, query, top_k=4):
        """
        Generates a RAG answer from retrieved article chunks.
        """

        retrieved_chunks = self.retrieve_context(query, top_k=top_k)

        if not retrieved_chunks:
            return {
                "mode": "fallback",
                "confidence": 0,
                "answer": (
                    "I could not find strong enough evidence in the article database "
                    "to generate a grounded answer for this query. Try using terms like "
                    "tomato, heirloom, production, organic, growth, fruit, soil, or agriculture."
                ),
                "sources": [],
                "retrieved_chunks": []
            }

        query_keywords = self.get_query_keywords(query)

        candidate_sentences = []

        for chunk in retrieved_chunks:
            sentences = self.sentence_split(chunk["text"])

            for sentence in sentences:
                sentence_lower = sentence.lower()

                keyword_score = 0

                for keyword in query_keywords:
                    if keyword in sentence_lower:
                        keyword_score += 1

                score = keyword_score + chunk["similarity"]

                candidate_sentences.append({
                    "sentence": sentence,
                    "score": score,
                    "DocID": chunk["DocID"],
                    "title": chunk["title"],
                    "file": chunk["file"],
                    "url": chunk["url"]
                })

        candidate_sentences.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        selected_sentences = []
        seen = set()

        for item in candidate_sentences:
            clean_sentence = item["sentence"]

            key = clean_sentence[:100].lower()

            if key in seen:
                continue

            selected_sentences.append(item)
            seen.add(key)

            if len(selected_sentences) == 4:
                break

        best_similarity = retrieved_chunks[0]["similarity"]
        confidence = round(min(best_similarity * 2.5, 1.0), 2)

        source_map = {}

        for chunk in retrieved_chunks:
            doc_id = chunk["DocID"]

            if doc_id not in source_map:
                source_map[doc_id] = {
                    "DocID": doc_id,
                    "title": chunk["title"],
                    "file": chunk["file"],
                    "url": chunk["url"],
                    "similarity": round(chunk["similarity"], 3)
                }

        sources = list(source_map.values())

        if selected_sentences:
            answer_body = " ".join(
                item["sentence"] for item in selected_sentences
            )
        else:
            answer_body = retrieved_chunks[0]["text"][:900] + "..."

        answer = (
            f"Based on the retrieved articles, the most relevant information suggests that "
            f"{answer_body}"
        )

        return {
            "mode": "rag",
            "confidence": confidence,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks
        }

    def get_index(self):
        """
        Returns cleaned index without sets/defaultdicts.
        """

        cleaned_index = {}

        for term, data in self.index.items():
            cleaned_index[term] = {
                "count": data["count"],
                "DocIDs": sorted(list(data["DocIDs"])),
                "entries": data["entries"],
                "doc_counts": dict(data["doc_counts"])
            }

        return cleaned_index

    def get_required_assignment_index(self):
        """
        Returns assignment-required format.
        """

        result = []

        for term in sorted(self.significant_terms):
            data = self.index.get(term)

            if data:
                doc_ids = sorted(list(data["DocIDs"]))
            else:
                doc_ids = []

            result.append({
                "term": term,
                "DocIDs": doc_ids
            })

        return result

    def get_articles_table(self):
        """
        Returns article metadata table.
        """

        return list(self.documents.values())

    def save_local_files(self):
        """
        Saves index outputs locally.
        """

        with open("full_query_index_with_rag.json", "w", encoding="utf-8") as file:
            json.dump(self.get_index(), file, indent=4, ensure_ascii=False)

        with open("required_index_format_with_rag.json", "w", encoding="utf-8") as file:
            json.dump(self.get_required_assignment_index(), file, indent=4, ensure_ascii=False)

        with open("articles_table_with_rag.json", "w", encoding="utf-8") as file:
            json.dump(self.get_articles_table(), file, indent=4, ensure_ascii=False)

        print("\n💾 Saved local files:")
        print("full_query_index_with_rag.json")
        print("required_index_format_with_rag.json")
        print("articles_table_with_rag.json")

    def save_index_to_firebase(self):
        """
        Saves the built inverted index to Firebase under search_index/
        Only called once — when the index does not exist in Firebase yet.
        """

        print("☁️ Saving index to Firebase for future use...")

        url = f"{self.firebase_url}/search_index.json"

        try:
            response = requests.put(url, json=self.get_index(), timeout=30)

            if response.status_code == 200:
                print(f"✅ Index saved to Firebase at: search_index/")
                print(f"   Terms saved: {len(self.get_index())}")
            else:
                print(f"⚠️ Firebase save returned status {response.status_code}: {response.text[:200]}")

        except Exception as e:
            print(f"❌ Failed to save index to Firebase: {e}")

    def load_index_from_firebase(self):
        """
        Tries to load the already-built index from Firebase.
        Returns True if loaded successfully, False if not found.

        Handles Firebase JSON quirks:
        - dicts with single entries may come back as lists
        - numeric keys may be returned as strings
        - doc_counts values need to be cast to int
        """

        url = f"{self.firebase_url}/search_index.json"

        try:
            response = requests.get(url, timeout=15)

            if response.status_code != 200:
                return False

            data = response.json()

            if not data or not isinstance(data, dict):
                return False

            for term, term_data in data.items():

                if not isinstance(term_data, dict):
                    continue

                self.index[term]["count"] = term_data.get("count", 0)

                # DocIDs — Firebase may return a dict with numeric keys instead of a list
                doc_ids_raw = term_data.get("DocIDs", [])
                if isinstance(doc_ids_raw, dict):
                    doc_ids_raw = list(doc_ids_raw.values())
                self.index[term]["DocIDs"] = set(str(d) for d in doc_ids_raw)

                # entries — Firebase may return a dict with numeric keys instead of a list
                entries_raw = term_data.get("entries", [])
                if isinstance(entries_raw, dict):
                    entries_raw = list(entries_raw.values())
                self.index[term]["entries"] = entries_raw if isinstance(entries_raw, list) else []

                # seen_titles — rebuilt from entries
                self.index[term]["seen_titles"] = set(
                    e["title"].strip().lower()
                    for e in self.index[term]["entries"]
                    if isinstance(e, dict) and "title" in e
                )

                # doc_counts — Firebase may return a list if only one entry exists
                doc_counts_raw = term_data.get("doc_counts", {})
                if isinstance(doc_counts_raw, list):
                    doc_counts_raw = {}
                if isinstance(doc_counts_raw, dict):
                    self.index[term]["doc_counts"] = defaultdict(
                        int, {str(k): int(v) for k, v in doc_counts_raw.items()}
                    )
                else:
                    self.index[term]["doc_counts"] = defaultdict(int)

            # Restore self.documents from entries
            for term_data in data.values():
                if not isinstance(term_data, dict):
                    continue
                entries_raw = term_data.get("entries", [])
                if isinstance(entries_raw, dict):
                    entries_raw = list(entries_raw.values())
                for entry in entries_raw:
                    if not isinstance(entry, dict):
                        continue
                    doc_id = entry.get("DocID")
                    if doc_id and doc_id not in self.documents:
                        self.documents[doc_id] = entry

            print(f"✅ Index loaded from Firebase.")
            print(f"   Terms: {len(data)} | Documents: {len(self.documents)}")
            return True

        except Exception as e:
            print(f"⚠️ Could not load index from Firebase: {e}")
            return False

    def run_all(self):
        """
        Singleton pipeline:
        1. Fetch articles from Firebase (always needed for RAG chunks)
        2. Check if index already exists in Firebase
           → YES: load it directly (fast, ~2 seconds)
           → NO:  build from articles + save to Firebase (runs once only)
        3. Always rebuild RAG chunks + vector store in memory
        """

        print("🚀 Starting Smart Tomato Care Search + RAG system...")

        # STEP 1: Fetch articles (always needed for RAG chunks)
        self.fetch_articles_from_firebase()

        # STEP 2: Singleton check — load or build index
        print("\n🔍 Checking if index already exists in Firebase...")
        index_loaded = self.load_index_from_firebase()

        if index_loaded:
            print("⚡ Index found in Firebase — skipping rebuild.")
        else:
            print("🔨 Index not found — building from articles...")

            for doc_id, article in self.articles_json.items():
                doc_id = str(doc_id)
                print(f"📄 Indexing DocID {doc_id}: {article.get('title', 'No Title')}")
                self.build_index_for_article(doc_id, article)

            self.save_local_files()
            self.save_index_to_firebase()

        # STEP 3: Always rebuild RAG chunks + vector store (in-memory only)
        print("\n🧠 Building RAG chunks and vector store...")
        self.build_rag_chunks()
        self.build_vector_store()

        print("\n✅ Search + RAG system is ready.")

In [ ]:
from collections import defaultdict


def fixed_load_index_from_firebase(self):
    """
    Loads the already-built search index from Firebase.

    Fixed version:
    - correctly restores doc_counts even if Firebase returns them as a list
    - restores entries
    - restores DocIDs
    - restores self.documents from entries
    """

    url = f"{self.firebase_url}/search_index.json"

    try:
        response = requests.get(url, timeout=15)

        if response.status_code != 200:
            print(f"⚠️ Firebase index load failed with status {response.status_code}")
            return False

        data = response.json()

        if not data or not isinstance(data, dict):
            print("⚠️ No valid search_index found in Firebase.")
            return False

        self.reset_index_data()

        for term, term_data in data.items():

            if not isinstance(term_data, dict):
                continue

            self.index[term]["count"] = int(term_data.get("count", 0))

            # ----------------------------
            # Restore DocIDs
            # ----------------------------
            doc_ids_raw = term_data.get("DocIDs", [])

            if isinstance(doc_ids_raw, dict):
                doc_ids_raw = list(doc_ids_raw.values())

            if isinstance(doc_ids_raw, list):
                self.index[term]["DocIDs"] = set(str(d) for d in doc_ids_raw if d is not None)
            else:
                self.index[term]["DocIDs"] = set()

            # ----------------------------
            # Restore entries
            # ----------------------------
            entries_raw = term_data.get("entries", [])

            if isinstance(entries_raw, dict):
                entries_raw = list(entries_raw.values())

            if isinstance(entries_raw, list):
                self.index[term]["entries"] = [
                    entry for entry in entries_raw
                    if isinstance(entry, dict)
                ]
            else:
                self.index[term]["entries"] = []

            # ----------------------------
            # Restore seen_titles from entries
            # ----------------------------
            self.index[term]["seen_titles"] = set(
                entry.get("title", "").strip().lower()
                for entry in self.index[term]["entries"]
                if isinstance(entry, dict)
            )

            # ----------------------------
            # Restore doc_counts
            # ----------------------------
            doc_counts_raw = term_data.get("doc_counts", {})

            fixed_doc_counts = {}

            if isinstance(doc_counts_raw, dict):
                for doc_id, count in doc_counts_raw.items():
                    try:
                        fixed_doc_counts[str(doc_id)] = int(count)
                    except:
                        pass

            elif isinstance(doc_counts_raw, list):
                # Firebase sometimes returns numeric-key objects as lists.
                # Example:
                # [None, 12, 5, 8]
                # means:
                # DocID "1" -> 12
                # DocID "2" -> 5
                # DocID "3" -> 8
                for doc_id, count in enumerate(doc_counts_raw):
                    if count is None:
                        continue

                    try:
                        fixed_doc_counts[str(doc_id)] = int(count)
                    except:
                        pass

            self.index[term]["doc_counts"] = defaultdict(int, fixed_doc_counts)

        # ----------------------------
        # Restore self.documents from entries
        # ----------------------------
        self.documents = {}

        for term_data in self.index.values():
            entries = term_data.get("entries", [])

            for entry in entries:
                if not isinstance(entry, dict):
                    continue

                doc_id = entry.get("DocID")

                if doc_id is None:
                    continue

                doc_id = str(doc_id)

                if doc_id not in self.documents:
                    self.documents[doc_id] = {
                        "DocID": doc_id,
                        "title": entry.get("title", "No Title"),
                        "file": entry.get("file", "Unknown File"),
                        "url": entry.get("url", "#")
                    }

        print("✅ Index loaded from Firebase.")
        print(f"   Terms: {len(self.index)} | Documents: {len(self.documents)}")
        print(f"   Usable doc_counts: {self.index_has_usable_doc_counts()}")

        return True

    except Exception as e:
        print(f"⚠️ Could not load index from Firebase: {e}")
        return False


def reset_index_data(self):
    """
    Clears the in-memory inverted index before rebuilding.
    """

    self.index = defaultdict(lambda: {
        "count": 0,
        "DocIDs": set(),
        "entries": [],
        "seen_titles": set(),
        "doc_counts": defaultdict(int)
    })

    self.documents = {}


def index_has_usable_doc_counts(self):
    """
    Checks if the loaded index has real per-document word counts.
    """

    current_index = self.get_index()

    for term, data in current_index.items():
        doc_counts = data.get("doc_counts", {})

        if not isinstance(doc_counts, dict):
            continue

        for count in doc_counts.values():
            try:
                if int(count) > 0:
                    return True
            except:
                continue

    return False


def fixed_run_all(self):
    """
    Fixed singleton pipeline.

    If Firebase has an old/broken search_index without doc_counts,
    this rebuilds the index from articles and saves the fixed version.
    """

    print("🚀 Starting Smart Tomato Care Search + RAG system...")

    # STEP 1: Fetch articles
    self.fetch_articles_from_firebase()

    # STEP 2: Try loading existing index
    print("\n🔍 Checking if index already exists in Firebase...")
    index_loaded = self.load_index_from_firebase()

    if index_loaded and self.index_has_usable_doc_counts():
        print("⚡ Index found in Firebase with usable doc_counts — skipping rebuild.")

    else:
        if index_loaded:
            print("⚠️ Firebase index exists, but doc_counts are missing or empty.")
            print("🔨 Rebuilding index from articles so indexed word-count ranking works...")
        else:
            print("🔨 Index not found — building from articles...")

        self.reset_index_data()

        for doc_id, article in self.articles_json.items():
            doc_id = str(doc_id)
            print(f"📄 Indexing DocID {doc_id}: {article.get('title', 'No Title')}")
            self.build_index_for_article(doc_id, article)

        self.save_local_files()
        self.save_index_to_firebase()

    # STEP 3: Always rebuild RAG chunks + vector store
    print("\n🧠 Building RAG chunks and vector store...")
    self.build_rag_chunks()
    self.build_vector_store()

    print("\n✅ Search + RAG system is ready.")


# Apply the fixes to the existing class
FirebaseRAGIndexService.load_index_from_firebase = fixed_load_index_from_firebase
FirebaseRAGIndexService.reset_index_data = reset_index_data
FirebaseRAGIndexService.index_has_usable_doc_counts = index_has_usable_doc_counts
FirebaseRAGIndexService.run_all = fixed_run_all

print("✅ FirebaseRAGIndexService patched successfully.")

✅ FirebaseRAGIndexService patched successfully.


In [ ]:
class QueryService:
    """
    Searches the index and connects the search engine to the RAG answer.
    """

    def __init__(self, index_service):
        self.index_service = index_service

    def get_document_metadata(self, doc_id, fallback_entry=None):
        """
        Gets document metadata safely.

        This is useful because sometimes the index is loaded from Firebase,
        so document data may come from the index entries.
        """

        doc_id = str(doc_id)

        document = self.index_service.documents.get(doc_id, {})

        if document:
            return document

        if fallback_entry:
            return {
                "DocID": doc_id,
                "title": fallback_entry.get("title", "No Title"),
                "file": fallback_entry.get("file", "Unknown File"),
                "url": fallback_entry.get("url", "#")
            }

        return {
            "DocID": doc_id,
            "title": "No Title",
            "file": "Unknown File",
            "url": "#"
        }

    def search(self, term):
        index = self.index_service.get_index()
        term = term.strip().lower()

        normalized_term = self.index_service.normalize_word_to_term(term)

        if normalized_term is None:
            return []

        return index.get(normalized_term, {}).get("entries", [])

    def search_with_score(self, term):
        """
        Single-term search.

        Returns:
        - total count across all documents
        - entries
        - count of this term in each document
        """

        index = self.index_service.get_index()
        term = term.strip().lower()

        normalized_term = self.index_service.normalize_word_to_term(term)

        if normalized_term is None or normalized_term not in index:
            return {
                "term": term,
                "count": 0,
                "entries": []
            }

        doc_counts = index[normalized_term].get("doc_counts", {})
        entries = []

        for entry in index[normalized_term].get("entries", []):
            new_entry = entry.copy()

            doc_id = str(new_entry.get("DocID", ""))
            new_entry["term_count"] = int(doc_counts.get(doc_id, 0))

            entries.append(new_entry)

        return {
            "term": normalized_term,
            "count": index[normalized_term]["count"],
            "entries": entries
        }

    def ranked_search(self, query):
        """
        Ranked document search.

        If the query contains words from the selected assignment index:
            score = indexed word count inside each document

        If the query contains no indexed words:
            score = semantic similarity from RAG/vector search
        """

        index = self.index_service.get_index()
        query_words = self.index_service.clean_and_tokenize(query)

        scores = {}
        used_index_terms = set()

        # -----------------------------
        # 1. Try indexed word-count search first
        # -----------------------------
        for word in query_words:
            normalized_term = self.index_service.normalize_word_to_term(word)

            if normalized_term is None:
                continue

            if normalized_term in used_index_terms:
                continue

            used_index_terms.add(normalized_term)

            if normalized_term not in index:
                continue

            doc_counts = index[normalized_term].get("doc_counts", {})

            if not doc_counts:
                continue

            entries = index[normalized_term].get("entries", [])
            entry_by_doc_id = {}

            for entry in entries:
                if not isinstance(entry, dict):
                    continue

                entry_doc_id = str(entry.get("DocID", ""))

                if entry_doc_id:
                    entry_by_doc_id[entry_doc_id] = entry

            for doc_id, count in doc_counts.items():
                doc_id = str(doc_id)

                try:
                    count = int(count)
                except:
                    continue

                if count <= 0:
                    continue

                fallback_entry = entry_by_doc_id.get(doc_id)
                document = self.get_document_metadata(doc_id, fallback_entry)

                if doc_id not in scores:
                    scores[doc_id] = {
                        "DocID": doc_id,
                        "title": document.get("title", "No Title"),
                        "file": document.get("file", "Unknown File"),
                        "url": document.get("url", "#"),
                        "score": 0,
                        "score_type": "indexed_word_count",
                        "matched_terms": set(),
                        "term_counts": {}
                    }

                scores[doc_id]["score"] += count
                scores[doc_id]["matched_terms"].add(normalized_term)
                scores[doc_id]["term_counts"][normalized_term] = count

        # -----------------------------
        # 2. Fallback to semantic search only if no indexed term matched
        # -----------------------------
        if not scores:
            retrieved_chunks = self.index_service.retrieve_context(query, top_k=5)

            for chunk in retrieved_chunks:
                doc_id = str(chunk["DocID"])
                similarity_score = round(float(chunk["similarity"]) * 100, 2)

                if doc_id not in scores:
                    scores[doc_id] = {
                        "DocID": doc_id,
                        "title": chunk.get("title", "No Title"),
                        "file": chunk.get("file", "Unknown File"),
                        "url": chunk.get("url", "#"),
                        "score": similarity_score,
                        "score_type": "semantic_similarity",
                        "matched_terms": set(["semantic-match"]),
                        "term_counts": {}
                    }

                else:
                    if similarity_score > scores[doc_id]["score"]:
                        scores[doc_id]["score"] = similarity_score

        ranked_results = []

        for doc_id, data in scores.items():
            ranked_results.append({
                "DocID": data["DocID"],
                "title": data["title"],
                "file": data["file"],
                "url": data["url"],
                "score": data["score"],
                "score_type": data["score_type"],
                "matched_terms": sorted(list(data["matched_terms"])),
                "term_counts": data["term_counts"]
            })

        ranked_results.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        return ranked_results

    def get_best_match(self, query):
        ranked_results = self.ranked_search(query)

        if not ranked_results:
            return None

        return ranked_results[0]

    def get_rag_answer(self, query, top_k=4):
        return self.index_service.generate_rag_answer(query, top_k=top_k)

In [ ]:
class ResultService:
    """
    Renders:
    1. RAG generated answer
    2. Ranked source documents

    Styled to match the Smart Tomato Care dashboard design.
    """

    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service

    def format_score_type(self, score_type):
        if score_type == "indexed_word_count":
            return "Indexed word-count score"

        if score_type == "semantic_similarity":
            return "Semantic similarity score"

        return "Score"

    def format_term_counts(self, term_counts):
        if not term_counts:
            return "No indexed word matched"

        parts = []

        for term, count in sorted(term_counts.items()):
            parts.append(f"{term}: {count}")

        return ", ".join(parts)

    def format_matched_terms(self, matched_terms):
        if not matched_terms:
            return "No matched terms"

        return ", ".join(matched_terms)

    def get_rag_answer_html(self, query):
        rag_result = self.query_service.get_rag_answer(query, top_k=4)

        answer = rag_result["answer"]
        confidence = rag_result["confidence"]
        mode = rag_result["mode"]
        sources = rag_result["sources"]

        if mode == "fallback":
            status_title = "AI Answer: Low Confidence"
            status_icon = "⚠️"
            badge_color = "#f59e0b"
            box_background = "#fff7ed"
            border_color = "#f59e0b"
        else:
            status_title = "AI Generated Answer"
            status_icon = "✨"
            badge_color = "#16a34a"
            box_background = "#ffffff"
            border_color = "#22c55e"

        html = f"""
        <div class="rag-answer-card" style="
            background: {box_background};
            border-left: 10px solid {border_color};
        ">
            <div class="rag-answer-top">
                <div>
                    <div class="rag-answer-title">
                        {status_icon} {status_title}
                    </div>
                    <div class="rag-answer-subtitle">
                        Generated from the most relevant research passages in Firebase.
                    </div>
                </div>

                <div class="confidence-badge" style="background:{badge_color};">
                    Confidence {confidence}
                </div>
            </div>

            <div class="rag-answer-text">
                {answer}
            </div>
        """

        if sources:
            html += """
            <div class="rag-source-box">
                <div class="rag-source-title">Sources used for this answer</div>
                <div class="rag-source-list">
            """

            for source in sources:
                html += f"""
                <a class="rag-source-pill" href="{source["url"]}" target="_blank">
                    📄 {source["file"]}
                    <span>similarity {source["similarity"]}</span>
                </a>
                """

            html += """
                </div>
            </div>
            """

        html += "</div>"

        return html

    def get_required_index_html(self):
        required_index = self.index_service.get_required_assignment_index()

        html = """
        <div class="dashboard-card">
            <h2 class="section-title">📚 Required Assignment Index</h2>
            <p class="section-subtitle">Each selected term and the documents where it appears.</p>

            <table class="green-table">
                <tr>
                    <th>Term</th>
                    <th>DocIDs</th>
                </tr>
        """

        for item in required_index:
            html += f"""
                <tr>
                    <td>{item["term"]}</td>
                    <td>{item["DocIDs"]}</td>
                </tr>
            """

        html += """
            </table>
        </div>
        """

        return html

    def get_result_html(self, term):
        """
        Single-word result page.

        Shows:
        - total word appearances
        - appearances in each document
        """

        result = self.query_service.search_with_score(term)

        entries = result["entries"]
        count = result["count"]

        if not entries:
            return f"""
            <div class="empty-result">
                ❌ No results found for <b>{term}</b>.
            </div>
            """

        html = f"""
        <div class="dashboard-card">
            <h2 class="section-title">🔍 Search Results for: {term}</h2>
            <p class="section-subtitle">Total word appearances: <b>{count}</b></p>
        """

        for idx, entry in enumerate(entries, start=1):
            title = entry.get("title", "No Title")
            pdf_file = entry.get("file", "#")
            article_url = entry.get("url", "#")
            doc_id = entry.get("DocID", "Unknown")
            term_count = entry.get("term_count", 0)

            html += f"""
            <div class="source-card">
                <div class="source-rank">{idx}</div>

                <div class="source-content">
                    <div class="source-title">DocID {doc_id} — {title}</div>

                    <a class="source-link" href="{article_url}" target="_blank">
                        📄 {pdf_file}
                    </a>

                    <div class="source-meta">
                        <span>Appearances in this document: <b>{term_count}</b></span>
                    </div>
                </div>
            </div>
            """

        html += "</div>"
        return html

    def get_ranked_search_html(self, query):
        rag_html = self.get_rag_answer_html(query)

        results = self.query_service.ranked_search(query)

        if not results:
            return rag_html + f"""
            <div class="empty-result">
                ❌ No matching file found for <b>{query}</b>.
            </div>
            """

        best = results[0]

        best_score_type = self.format_score_type(best.get("score_type"))
        best_matched_terms = self.format_matched_terms(best.get("matched_terms", []))
        best_term_counts = self.format_term_counts(best.get("term_counts", {}))

        html = rag_html

        html += f"""
        <div class="dashboard-card">
            <h2 class="section-title">📚 Source Documents</h2>
            <p class="section-subtitle">Ranked documents related to: <b>{query}</b></p>

            <div class="best-document-card">
                <div class="best-document-label">Best Matching Document</div>

                <div class="best-document-title">
                    <a href="{best["url"]}" target="_blank">
                        {best["file"]}
                    </a>
                </div>

                <div class="best-document-grid">
                    <div>
                        <span>DocID</span>
                        <b>{best["DocID"]}</b>
                    </div>

                    <div>
                        <span>Score</span>
                        <b>{best["score"]}</b>
                    </div>

                    <div>
                        <span>Score Type</span>
                        <b>{best_score_type}</b>
                    </div>

                    <div>
                        <span>Matched Terms</span>
                        <b>{best_matched_terms}</b>
                    </div>

                    <div>
                        <span>Word Counts</span>
                        <b>{best_term_counts}</b>
                    </div>
                </div>
            </div>

            <h3 class="small-section-title">All Ranked Documents</h3>
        """

        for idx, result in enumerate(results, start=1):
            score_type = self.format_score_type(result.get("score_type"))
            matched_terms = self.format_matched_terms(result.get("matched_terms", []))
            term_counts = self.format_term_counts(result.get("term_counts", {}))

            html += f"""
            <div class="source-card">
                <div class="source-rank">{idx}</div>

                <div class="source-content">
                    <div class="source-title">
                        DocID {result["DocID"]} — {result["title"]}
                    </div>

                    <a class="source-link" href="{result["url"]}" target="_blank">
                        📄 {result["file"]}
                    </a>

                    <div class="source-meta">
                        <span>Score: <b>{result["score"]}</b></span>
                        <span>Score Type: <b>{score_type}</b></span>
                        <span>Matched Terms: <b>{matched_terms}</b></span>
                        <span>Word Counts: <b>{term_counts}</b></span>
                    </div>
                </div>
            </div>
            """

        html += "</div>"

        return html

In [ ]:
firebase_url = "https://smart-tomato-care-default-rtdb.firebaseio.com/"

index_service = FirebaseRAGIndexService(
    firebase_url=firebase_url,
    articles_path="articles_json",
    chunk_size=900,
    chunk_overlap=150,
    min_similarity=0.04
)

query_service = QueryService(index_service)
result_service = ResultService(index_service, query_service)

index_service.run_all()

🚀 Starting Smart Tomato Care Search + RAG system...
⬇️ Reading articles from Firebase:
https://smart-tomato-care-default-rtdb.firebaseio.com/articles_json.json
✅ Loaded 5 articles from Firebase.

🔍 Checking if index already exists in Firebase...
✅ Index loaded from Firebase.
   Terms: 19 | Documents: 5
   Usable doc_counts: True
⚡ Index found in Firebase with usable doc_counts — skipping rebuild.

🧠 Building RAG chunks and vector store...
✅ Created 88 text chunks for RAG.
✅ TF-IDF vector store is ready.

✅ Search + RAG system is ready.


In [ ]:
# =====================================================
# GARDEN SEARCH TAB SCREEN
# =====================================================

import ipywidgets as widgets
from IPython.display import HTML, display

# CSS design for the search tab
search_design_css = widgets.HTML("""
<style>
    .garden-search-panel {
        background: linear-gradient(135deg, #ffffff 0%, #f0fbf5 100%);
        border: 1px solid #dceee5;
        border-radius: 26px;
        padding: 36px 42px;
        box-shadow: 0 15px 40px rgba(15, 118, 59, 0.08);
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 25px auto;
        width: 92%;
    }

    .garden-search-title {
        color: #14532d;
        font-size: 42px;
        font-weight: 900;
        margin-bottom: 4px;
    }

    .garden-search-subtitle {
        color: #64748b;
        font-size: 18px;
        margin-bottom: 28px;
    }

    .garden-search-input input {
        border: 2px solid #bbf7d0 !important;
        border-radius: 16px !important;
        padding: 16px 20px !important;
        font-size: 17px !important;
        color: #0f172a !important;
        background: white !important;
        box-shadow: 0 8px 20px rgba(15, 118, 59, 0.06) !important;
    }

    .garden-search-input input:focus {
        border-color: #22c55e !important;
        box-shadow: 0 0 0 4px rgba(34, 197, 94, 0.15) !important;
        outline: none !important;
    }

    .garden-search-button button {
        background: linear-gradient(180deg, #22c55e 0%, #166534 100%) !important;
        color: white !important;
        border: none !important;
        border-radius: 16px !important;
        font-size: 17px !important;
        font-weight: 800 !important;
        height: 52px !important;
        box-shadow: 0 10px 24px rgba(22, 101, 52, 0.28) !important;
    }

    .garden-search-button button:hover {
        background: linear-gradient(180deg, #16a34a 0%, #14532d 100%) !important;
    }

    .rag-answer-card {
        background: white;
        border-left: 10px solid #22c55e;
        border-radius: 24px;
        padding: 30px;
        margin-top: 28px;
        box-shadow: 0 12px 34px rgba(15, 23, 42, 0.08);
        border-top: 1px solid #e5e7eb;
        border-right: 1px solid #e5e7eb;
        border-bottom: 1px solid #e5e7eb;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }

    .rag-answer-top {
        display: flex;
        align-items: flex-start;
        justify-content: space-between;
        gap: 20px;
        margin-bottom: 18px;
    }

    .rag-answer-title {
        color: #14532d;
        font-size: 30px;
        font-weight: 900;
    }

    .rag-answer-subtitle {
        color: #64748b;
        font-size: 16px;
        margin-top: 4px;
    }

    .confidence-badge {
        color: white;
        background: #16a34a;
        padding: 10px 18px;
        border-radius: 999px;
        font-weight: 800;
        white-space: nowrap;
        box-shadow: 0 8px 18px rgba(0,0,0,0.12);
    }

    .rag-answer-text {
        color: #111827;
        font-size: 17px;
        line-height: 1.75;
        background: #f8fafc;
        padding: 20px;
        border-radius: 18px;
        border: 1px solid #e5e7eb;
    }

    .rag-source-box {
        margin-top: 18px;
        background: #ecfdf5;
        padding: 18px;
        border-radius: 18px;
        border-left: 5px solid #22c55e;
    }

    .rag-source-title {
        color: #14532d;
        font-weight: 900;
        margin-bottom: 12px;
        font-size: 17px;
    }

    .rag-source-list {
        display: flex;
        flex-wrap: wrap;
        gap: 10px;
    }

    .rag-source-pill {
        text-decoration: none;
        background: white;
        border: 1px solid #bbf7d0;
        color: #166534;
        padding: 10px 14px;
        border-radius: 999px;
        font-size: 14px;
        font-weight: 700;
    }

    .dashboard-card {
        background: white;
        border-radius: 24px;
        padding: 30px;
        margin-top: 28px;
        box-shadow: 0 12px 34px rgba(15, 23, 42, 0.08);
        border: 1px solid #e5e7eb;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }

    .section-title {
        color: #14532d;
        font-size: 30px;
        font-weight: 900;
        margin: 0;
    }

    .section-subtitle {
        color: #64748b;
        font-size: 16px;
        margin-top: 6px;
        margin-bottom: 22px;
    }

    .best-document-card {
        background: linear-gradient(135deg, #f0fdf4 0%, #ffffff 100%);
        border-left: 8px solid #22c55e;
        border-radius: 22px;
        padding: 24px;
        margin-bottom: 22px;
        box-shadow: 0 8px 22px rgba(22, 101, 52, 0.08);
    }

    .best-document-label {
        color: #16a34a;
        font-size: 16px;
        font-weight: 900;
        margin-bottom: 6px;
    }

    .best-document-title a {
        color: #0f172a;
        text-decoration: none;
        font-size: 24px;
        font-weight: 900;
    }

    .best-document-grid {
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 14px;
        margin-top: 18px;
    }

    .best-document-grid div {
        background: white;
        border-radius: 16px;
        padding: 14px;
        border: 1px solid #e5e7eb;
    }

    .best-document-grid span {
        display: block;
        color: #64748b;
        font-size: 13px;
        margin-bottom: 5px;
    }

    .best-document-grid b {
        color: #14532d;
        font-size: 16px;
    }

    .source-card {
        display: flex;
        gap: 18px;
        background: #ffffff;
        border: 1px solid #e5e7eb;
        border-radius: 20px;
        padding: 18px;
        margin-bottom: 14px;
        box-shadow: 0 8px 20px rgba(15, 23, 42, 0.05);
    }

    .source-rank {
        width: 46px;
        height: 46px;
        background: linear-gradient(180deg, #22c55e 0%, #166534 100%);
        color: white;
        border-radius: 50%;
        display: flex;
        align-items: center;
        justify-content: center;
        font-weight: 900;
        font-size: 18px;
        flex-shrink: 0;
    }

    .source-content {
        flex: 1;
    }

    .source-title {
        color: #0f172a;
        font-size: 18px;
        font-weight: 900;
        margin-bottom: 8px;
    }

    .source-link {
        color: #166534;
        text-decoration: none;
        font-weight: 700;
    }

    .source-meta {
        display: flex;
        flex-wrap: wrap;
        gap: 16px;
        color: #64748b;
        margin-top: 10px;
        font-size: 14px;
    }

    .empty-result {
        background: #fff7ed;
        border-left: 7px solid #f97316;
        color: #7c2d12;
        padding: 18px;
        border-radius: 18px;
        font-size: 16px;
        margin-top: 20px;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }

    .loading-message {
        text-align: center;
        color: #166534;
        font-size: 17px;
        font-weight: 700;
        margin: 18px 0;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
</style>
""")

search_title = widgets.HTML("""
<div class="garden-search-panel">
    <div class="garden-search-title">Garden Research Search 🍅</div>
    <div class="garden-search-subtitle">
        Ask questions about heirloom tomatoes, organic production, growth, fruit quality, and agriculture.
    </div>
</div>
""")

search_input = widgets.Text(
    placeholder="Ask the tomato research engine...",
    layout=widgets.Layout(width="680px")
)
search_input.add_class("garden-search-input")

search_btn = widgets.Button(
    description="Search",
    layout=widgets.Layout(width="140px")
)
search_btn.add_class("garden-search-button")

search_output = widgets.Output()

def perform_search(b=None):
    with search_output:
        search_output.clear_output()

        query_val = search_input.value.strip()

        if query_val:
            display(HTML("<div class='loading-message'>🌿 Retrieving context and generating answer...</div>"))
            display(HTML(result_service.get_ranked_search_html(query_val)))
        else:
            display(HTML("""
            <div class="empty-result">
                Type a question first. Example:
                <b>How does organic production affect heirloom tomato growth?</b>
            </div>
            """))

search_btn.on_click(perform_search)

try:
    search_input.on_submit(lambda x: perform_search())
except:
    pass

search_bar = widgets.HBox(
    [search_input, search_btn],
    layout=widgets.Layout(
        justify_content="center",
        margin="0 0 20px 0"
    )
)

search_screen = widgets.VBox([
    search_design_css,
    search_title,
    search_bar,
    search_output
])

## 🤖 Chatbot

In [ ]:
# =====================================================
# FEATURE: APP HELP CHATBOT
# =====================================================

import ipywidgets as widgets
from IPython.display import display
import google.generativeai as genai

# =====================================================
# 1) GEMINI CONFIGURATION
# =====================================================
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"
genai.configure(api_key=GEMINI_API_KEY)

chat_model = genai.GenerativeModel("gemini-2.5-flash")
chat_session = chat_model.start_chat(history=[])

# =====================================================
# 2) STYLE
# =====================================================
chat_style = widgets.HTML("""
<style>
.chat-shell {
    background: linear-gradient(135deg, #f8fafc 0%, #f0fdf4 100%);
    border-radius: 28px;
    padding: 24px;
    border: 1px solid #e5e7eb;
    font-family: Arial, sans-serif;
}

.chat-title {
    font-size: 24px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 6px;
}

.chat-subtitle {
    font-size: 13px;
    color: #64748b;
    margin-bottom: 18px;
}

.chat-box {
    background: #f8fafc;
    border-radius: 18px;
    padding: 14px;
    min-height: 220px;
    max-height: 320px;
    overflow-y: auto;
    border: 1px solid #e5e7eb;
}

.user-msg {
    background: #dcfce7;
    color: #14532d;
    padding: 10px 12px;
    border-radius: 14px;
    margin-bottom: 10px;
    font-size: 13px;
    font-weight: 600;
}

.bot-msg {
    background: #ffffff;
    color: #334155;
    padding: 10px 12px;
    border-radius: 14px;
    margin-bottom: 10px;
    font-size: 13px;
    line-height: 1.6;
    border-left: 4px solid #22c55e;
}

.send-btn-style {
    background: linear-gradient(180deg, #22c55e 0%, #166534 100%) !important;
    color: white !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: none !important;
}

.clear-btn-style {
    background: white !important;
    color: #b91c1c !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: 1px solid #fecaca !important;
}
</style>
""")

# =====================================================
# 3) UI COMPONENTS
# =====================================================
title = widgets.HTML("""
<div class="chat-title">App Help Chatbot 🤖</div>
<div class="chat-subtitle">
Ask how to use the application, where to find features, and how each screen works.
</div>
""")

chat_area = widgets.HTML("""
<div class="chat-box">
    <div class="bot-msg">
        🤖 Hello! I am your Tomato Assistant.<br>
        I can help you navigate this app <b>and</b> answer questions about
        🍅 <b>tomato diseases</b> and <b>tomato growth</b>.<br>
        What would you like to know?
    </div>
</div>
""")

question_input = widgets.Textarea(
    placeholder="Ask about the app...",
    layout=widgets.Layout(width="100%", height="80px")
)

btn_send = widgets.Button(
    description="Ask",
    icon="paper-plane",
    layout=widgets.Layout(width="110px", height="42px")
)
btn_send.add_class("send-btn-style")

btn_clear = widgets.Button(
    description="Clear",
    icon="trash",
    layout=widgets.Layout(width="110px", height="42px")
)
btn_clear.add_class("clear-btn-style")

buttons_row = widgets.HBox(
    [btn_send, btn_clear],
    layout=widgets.Layout(gap="10px", justify_content="flex-end")
)

chat_history_html = """
<div class="chat-box">
    <div class="bot-msg">
        🤖 Hello! I am your Tomato Assistant.<br>
        I can help you navigate this app <b>and</b> answer questions about
        🍅 <b>tomato diseases</b> and <b>tomato growth</b>.<br>
        What would you like to know?
    </div>
"""

# =====================================================
# 4) CHATBOT LOGIC
# =====================================================
def ask_gemini(question):
    prompt = f"""
   You are the helpful assistant of a Smart Tomato Plant Monitoring application.

You can help with TWO types of questions:

=== APP NAVIGATION HELP ===
The application has these screens:

1. Plant Scanner:
   - Upload a tomato plant image.
   - Analyze the image using AI.
   - Shows diagnosis, detected conditions, and care recommendations.

2. IoT Data:
   - Shows real-time sensor readings from the cloud.
   - Includes temperature, humidity, and soil moisture.
   - Shows latest values, averages, min/max, and feed details.

3. Dashboard:
   - Shows a visual summary of the plant condition.
   - Includes charts, sensor trends, and general plant status.

4. RAG Search:
   - Answers questions using academic articles.
   - Best for scientific questions or article-based explanations.

=== TOMATO EXPERTISE ===
You are also an expert in:
- Tomato diseases: early blight, late blight, bacterial spot, septoria leaf spot, mosaic virus,
  leaf mold, target spot, spider mites, yellow leaf curl virus, and more.
- Symptoms identification, causes, and treatment for each disease.
- Tomato growth stages: germination, seedling, vegetative, flowering, fruiting, and ripening.
- Optimal growing conditions: temperature, humidity, light, watering, fertilization, and soil pH.
- Pest management and organic/chemical treatment options.
- Prevention strategies and best practices for healthy tomato plants.

=== RULES ===
- For app-related questions: guide the user to the correct screen.
- For tomato disease or growth questions: provide clear, accurate, and helpful answers.
- For unrelated questions, politely say: "I can help with app navigation and tomato-related questions. Please ask me about those topics!"
- Keep answers clear, concise, and practical.
- Use English only.

    User question:
    {question}
    """

    response = chat_session.send_message(prompt)
    return response.text.strip()

def on_send_click(_):
    global chat_history_html

    question = question_input.value.strip()
    if not question:
        return

    chat_history_html += f"""
    <div class="user-msg">👤 {question}</div>
    """

    chat_area.value = chat_history_html + """
    <div class="bot-msg">⏳ Thinking...</div>
    </div>
    """

    question_input.value = ""

    try:
        answer = ask_gemini(question)
        answer = answer.replace("\n", "<br>")

        chat_history_html += f"""
        <div class="bot-msg">🤖 {answer}</div>
        """

    except Exception as e:
        chat_history_html += f"""
        <div class="bot-msg">❌ Error: {e}</div>
        """

    chat_area.value = chat_history_html + "</div>"

def on_clear_click(_):
    global chat_history_html

    chat_history_html = """
    <div class="chat-box">
        <div class="bot-msg">
        🤖 Hello! I am your Tomato Assistant.<br>
        I can help you navigate this app <b>and</b> answer questions about
        🍅 <b>tomato diseases</b> and <b>tomato growth</b>.<br>
        What would you like to know?
        </div>
    """

    chat_area.value = chat_history_html + "</div>"
    question_input.value = ""

btn_send.on_click(on_send_click)
btn_clear.on_click(on_clear_click)

# =====================================================
# 5) CHAT SCREEN
# =====================================================
chat_screen = widgets.VBox(
    [
        chat_style,
        widgets.VBox(
            [title, chat_area, question_input, buttons_row],
            layout=widgets.Layout(width="100%")
        )
    ],
    layout=widgets.Layout(width="100%")
)

chat_screen.add_class("chat-shell")

# =====================================================
# 6) FLOATING / COLLAPSIBLE CHATBOT
# =====================================================
chat_open = False

chat_toggle_btn = widgets.Button(
    description="💬 Chat",
    layout=widgets.Layout(width="120px", height="46px")
)
chat_toggle_btn.add_class("send-btn-style")

chat_container = widgets.VBox(
    [chat_screen],
    layout=widgets.Layout(display="none", width="430px")
)

floating_chat = widgets.VBox(
    [chat_toggle_btn, chat_container],
    layout=widgets.Layout(
        align_items="flex-end",
        width="100%",
        margin="20px 0 0 0"
    )
)

def toggle_chat(_):
    global chat_open
    chat_open = not chat_open

    if chat_open:
        chat_container.layout.display = "flex"
        chat_toggle_btn.description = "✖️ Close Chat"
    else:
        chat_container.layout.display = "none"
        chat_toggle_btn.description = "💬 Chat"

chat_toggle_btn.on_click(toggle_chat)

#def wrap_with_chatbot(screen_content):
#    return widgets.VBox([
#        screen_content,
#        floating_chat
#    ])

## 🧠 Tomato AI Analysis

In [ ]:
# =====================================================
# BACKEND LOGIC - TOMATO AI ANALYSIS
# =====================================================

import io
from PIL import Image
from transformers import pipeline

print("Initializing AI backend...")

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else -1
except:
    DEVICE = -1

# General image classifier: checks if image is not tomato-like
identity_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    device=DEVICE
)

# Plant disease classifier
try:
    disease_classifier_1 = pipeline(
        "image-classification",
        model="surprisedPikachu007/tomato-disease-detection",
        device=DEVICE
    )

    disease_classifier_2 = pipeline(
        "image-classification",
        model="wellCh4n/tomato-leaf-disease-classification-vit",
        device=DEVICE
    )

    HAS_DISEASE_MODEL = True

except Exception as e:
    print("Disease model could not load:", e)
    HAS_DISEASE_MODEL = False


def generate_simple_advice(plant_name, health_status, disease_info):
    return """
- Remove damaged or infected leaves.
- Keep the tomato plant in good sunlight.
- Avoid overwatering the soil.
- Monitor leaves and fruits for new symptoms.
"""


def run_analysis_logic(image_bytes: bytes):
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    identity_results = identity_classifier(image)
    identity_top = identity_results[0]

    detected_label = identity_top.get("label", "Unknown").lower()
    id_conf = float(identity_top.get("score", 0))

    # Reject clearly non-tomato images
    not_tomato_keywords = [
        "cucumber", "cuke", "banana", "apple", "orange",
        "carrot", "pepper", "potato", "onion", "grape",
        "dog", "cat", "car", "person"
    ]

    if any(word in detected_label for word in not_tomato_keywords) and id_conf >= 0.50:
        return {
            "name": identity_top.get("label", "Unknown"),
            "id_conf": f"{id_conf:.1%}",
            "health": "Not a tomato image",
            "health_conf": "—",
            "diseases": ["Image validation failed."],
            "advice": "Please upload a clear image of tomato leaves, tomato fruit, or a tomato plant.",
            "is_tomato": False
        }

    plant_name = "Tomato Plant"

    if HAS_DISEASE_MODEL:
        results_1 = disease_classifier_1(image)
        results_2 = disease_classifier_2(image)

        top1 = results_1[0]
        top2 = results_2[0]

        label1 = top1.get("label", "Unknown")
        score1 = float(top1.get("score", 0))

        label2 = top2.get("label", "Unknown")
        score2 = float(top2.get("score", 0))

        if label1.lower() == label2.lower():
            final_label = label1
            final_score = (score1 + score2) / 2
        else:
            if score1 >= score2:
                final_label = label1
                final_score = score1
            else:
                final_label = label2
                final_score = score2

        disease_list_str = f"{final_label} ({final_score:.1%})"

        if "healthy" in final_label.lower():
            health_status = "Healthy"
        else:
            health_status = "Possible Disease Detected"

        health_conf = final_score


    else:
        disease_list_str = "Disease model unavailable."
        health_status = "Unknown"
        health_conf = 0

    advice = generate_simple_advice(
        plant_name,
        health_status,
        disease_list_str
    )

    return {
        "name": plant_name,
        "id_conf": f"{id_conf:.1%}",
        "health": health_status,
        "health_conf": f"{health_conf:.1%}",
        "diseases": [disease_list_str],
        "advice": advice,
        "is_tomato": True
    }

print("✅ Backend Ready.")

Initializing AI backend...


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

✅ Backend Ready.


## 📸 Plant Scanner Screen

In [ ]:
# =====================================================
# SCREEN 1: TOMATO PLANT IMAGE UPLOAD - UPDATED DESIGN
# =====================================================

import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display
import re

# =====================================================
# 1) CSS STYLE
# =====================================================
plant_ui_style = widgets.HTML("""
<style>
.plant-screen {
    font-family: Arial, sans-serif;
}

.plant-shell {
    background: linear-gradient(135deg, #f8fafc 0%, #f0fdf4 100%);
    border-radius: 30px;
    padding: 34px;
    border: 1px solid #e5e7eb;
}

.top-title {
    font-size: 36px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 6px;
}

.top-subtitle {
    font-size: 15px;
    color: #64748b;
    margin-bottom: 28px;
}

.left-card, .right-card {
    background: white;
    border-radius: 26px;
    padding: 28px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 14px 34px rgba(15,23,42,0.07);
}

.upload-zone {
    position: relative !important;
    border: 2px dashed #86efac;
    border-radius: 24px;
    background:
      radial-gradient(circle at top left, rgba(34,197,94,0.13), transparent 35%),
      radial-gradient(circle at bottom right, rgba(239,68,68,0.10), transparent 38%),
      linear-gradient(180deg,#f0fdf4,#ffffff);

    min-height: 430px;

    display: flex;
    align-items: center;
    justify-content: center;
}

.upload-zone:hover {
    border-color: #22c55e;
    box-shadow: 0 14px 30px rgba(34,197,94,0.16);
}

.real-upload {
    position: absolute !important;
    inset: 0 !important;
    width: 100% !important;
    height: 100% !important;
    opacity: 0 !important;
    z-index: 10 !important;
    cursor: pointer !important;
}

.analyze-btn-new {
    background: linear-gradient(180deg, #22c55e 0%, #166534 100%) !important;
    color: white !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: none !important;
}

.reset-btn-new {
    background: white !important;
    color: #b91c1c !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: 1px solid #fecaca !important;
}

.tip-box {
    background: #f8fafc;
    border-radius: 16px;
    padding: 14px;
    margin-bottom: 12px;
    font-size: 13px;
    color: #334155;
    border-left: 4px solid #22c55e;
}

.result-card {
    background: #ffffff;
    border-radius: 20px;
    padding: 22px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 10px 25px rgba(15,23,42,0.06);
    margin-top: 20px;
}

.result-title {
    font-size: 18px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 12px;
}

.result-box {
    background: #f8fafc;
    border-left: 4px solid #22c55e;
    padding: 12px;
    border-radius: 10px;
    font-size: 13px;
    line-height: 1.6;
}
</style>
""")

# =====================================================
# 2) HEADER
# =====================================================
header = widgets.HTML("""
<div>
  <div class="top-title">Tomato Plant Health Scanner 🍅</div>

  <div class="top-subtitle">
    Upload or capture a clear tomato plant image to analyze its health condition using AI.
  </div>
</div>
""")

# =====================================================
# 3) REAL FILE UPLOAD
# =====================================================
file_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False
)

file_uploader.add_class("real-upload")

upload_visual = widgets.VBox([
    widgets.HTML("""
    <div style="font-size:58px;text-align:center;margin-bottom:22px;">🍅🌿🍃</div>

    <div style="
        width:110px;
        height:110px;
        border-radius:50%;
        background:#dcfce7;
        display:flex;
        align-items:center;
        justify-content:center;
        margin:0 auto 28px auto;
        box-shadow:0 12px 28px rgba(34,197,94,0.25);
        border:5px solid white;">

        <span style="font-size:52px;">📷</span>
    </div>

    <div style='font-size:28px;font-weight:900;color:#14532d;text-align:center;margin-bottom:12px;'>
        Upload Tomato Plant Image
    </div>

    <div style='font-size:15px;color:#64748b;text-align:center;line-height:1.7;'>
        Choose a clear photo of tomato
    </div>

    <div style='margin-top:24px;text-align:center;'>

        <span style='background:#fee2e2;color:#991b1b;
        padding:9px 16px;border-radius:999px;
        font-size:13px;font-weight:800;margin-right:8px;'>

        🍅 Tomato

    </div>
    """)
], layout=Layout(
    align_items="center",
    justify_content="center",
    width="100%",
    height="100%"
))

upload_zone = widgets.Box(
    [upload_visual, file_uploader],
    layout=Layout(width="100%", position="relative")
)

upload_zone.add_class("upload-zone")

# =====================================================
# 4) IMAGE PREVIEW + BUTTONS
# =====================================================
img_preview = widgets.Image(
    layout=Layout(
        width="100%",
        height="340px",
        object_fit="contain",
        border_radius="18px",
        border="1px solid #e5e7eb",
        margin="0 0 16px 0"
    )
)

btn_analyze = widgets.Button(
    description="Analyze Tomato Plant",
    icon="search",
    layout=Layout(width="100%", height="52px")
)

btn_analyze.add_class("analyze-btn-new")

btn_reset = widgets.Button(
    description="Choose Another Image",
    icon="refresh",
    layout=Layout(width="100%", height="48px", margin="10px 0 0 0")
)

btn_reset.add_class("reset-btn-new")

preview_area = widgets.VBox(
    [img_preview, btn_analyze, btn_reset],
    layout=Layout(display="none")
)

left_card = widgets.VBox(
    [upload_zone, preview_area],
    layout=Layout(width="70%")
)

left_card.add_class("left-card")

# =====================================================
# 5) PHOTO GUIDELINES
# =====================================================
guidelines = widgets.VBox([

    widgets.HTML("""

    <div style='font-size:24px;
    font-weight:900;
    color:#14532d;
    margin-bottom:18px;'>

    🍅 Tomato Photo Guidelines

    </div>
    """),

    widgets.HTML("<div class='tip-box'>✅ Place the tomato plant in the center of the image.</div>"),

    widgets.HTML("<div class='tip-box'>✅ Use bright natural light.</div>"),

    widgets.HTML("<div class='tip-box'>✅ Focus on leaves, fruit, spots, or damaged areas.</div>"),

    widgets.HTML("<div class='tip-box'>✅ Avoid blurry photos and dark backgrounds.</div>"),

    widgets.HTML("<div class='tip-box'>✅ Take the image from a close but clear distance.</div>")

], layout=Layout(width="28%"))

guidelines.add_class("right-card")

results_output = widgets.Output()

# =====================================================
# 6) HELPER FUNCTIONS
# =====================================================
def get_uploaded_content(uploader):

    vals = uploader.value

    if isinstance(vals, tuple) and vals:
        return vals[0].get("content", b"")

    if isinstance(vals, dict) and vals:
        return vals[list(vals.keys())[0]].get("content", b"")

    return b""

def clear_uploader(uploader):

    try:
        uploader.value.clear()
    except:
        pass

    for v in ({}, (), None):
        try:
            uploader.value = v
        except:
            pass

# =====================================================
# 7) EVENTS
# =====================================================
def on_upload_change(change):

    content = get_uploaded_content(file_uploader)

    if not content:
        return

    img_preview.value = content

    upload_zone.layout.display = "none"

    preview_area.layout.display = "flex"

    results_output.clear_output()

def on_reset_click(_):

    clear_uploader(file_uploader)

    img_preview.value = b""

    upload_zone.layout.display = "flex"

    preview_area.layout.display = "none"

    results_output.clear_output()

def on_analyze_click(_):

    if "run_analysis_logic" not in globals():

        with results_output:
            results_output.clear_output()
            print("❌ Please run the backend logic block first.")

        return

    if not img_preview.value:

        with results_output:
            results_output.clear_output()
            print("❌ Please upload an image first.")

        return

    btn_analyze.disabled = True
    btn_analyze.description = "Analyzing..."

    with results_output:

        results_output.clear_output()

        display(widgets.HTML("""
        <div style='padding:20px;color:#64748b;'>
        🔄 Running AI analysis...
        </div>
        """))

    try:

        results = run_analysis_logic(img_preview.value)

        diseases = results.get("diseases", [])

        if isinstance(diseases, list):
            diseases_html = "".join(f"<div>• {d}</div>" for d in diseases)

        else:
            diseases_html = f"<div>• {diseases}</div>"

        raw_advice = str(results.get("advice", ""))

        advice_html = ""

        for line in raw_advice.split("\\n"):

            clean = re.sub(r"^[\\d\\.\\-\\*]+\\s*", "", line.strip())

            if len(clean) > 3:
                advice_html += f"<div>• {clean}</div>"

        if not advice_html:
            advice_html = "<div>• No advice available.</div>"

        html = f"""

        <div class="result-card">

            <div class="result-title">
            AI Diagnosis Result
            </div>

            <div style="margin-bottom:12px;">

                <span style="
                background:#dcfce7;
                color:#166534;
                padding:7px 12px;
                border-radius:999px;
                font-size:12px;
                font-weight:800;">

                🌱 {results.get('name','Unknown')} - {results.get('id_conf','—')}

                </span>

                <span style="
                background:#dbeafe;
                color:#1e40af;
                padding:7px 12px;
                border-radius:999px;
                font-size:12px;
                font-weight:800;">

                ❤️ {results.get('health','Unknown')} - {results.get('health_conf','—')}

                </span>

            </div>

            <div style="font-weight:900;color:#334155;margin:12px 0 6px 0;">
            Detected Conditions
            </div>

            <div class="result-box">
            {diseases_html}
            </div>

            <div style="font-weight:900;color:#334155;margin:14px 0 6px 0;">
            Care Recommendations
            </div>

            <div class="result-box">
            {advice_html}
            </div>

        </div>
        """

        with results_output:

            results_output.clear_output()

            display(widgets.HTML(html))

    except Exception as e:

        with results_output:

            results_output.clear_output()

            print(f"Analysis Error: {e}")

    finally:

        btn_analyze.disabled = False

        btn_analyze.description = "Analyze Tomato Plant"

file_uploader.observe(on_upload_change, names="value")

btn_reset.on_click(on_reset_click)

btn_analyze.on_click(on_analyze_click)

# =====================================================
# 8) FINAL LAYOUT
# =====================================================
main_row = widgets.HBox(
    [left_card, guidelines],
    layout=Layout(
        gap="26px",
        align_items="stretch",
        justify_content="center"
    )
)

screen = widgets.VBox(
    [plant_ui_style, header, main_row, results_output],
    layout=Layout(width="98%", margin="10px auto")
)

screen.add_class("plant-shell")
screen.add_class("plant-screen")


## 🌱 IoT Data Screen

In [ ]:
# =====================================================
# SCREEN 2: IOT SENSOR DATA SAMPLING + SAVE TO FIREBASE
# Dynamic history amount + cards + feed details + DB save
# =====================================================

import requests
import json
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display
from datetime import datetime
import pandas as pd

# =====================================================
# 1) CLOUD + FIREBASE CONFIG
# =====================================================
BASE_URL = "https://server-cloud-v645.onrender.com/"

FIREBASE_DB_URL = "https://smart-tomato-care-default-rtdb.firebaseio.com"
SENSOR_HISTORY_PATH = f"{FIREBASE_DB_URL}/sensor_history.json"

# =====================================================
# 2) FETCH FROM RENDER SERVER
# =====================================================
def fetch_feed_samples(feed, limit=10):
    try:
        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": limit},
            timeout=90
        )

        data = response.json()

        if "data" in data:
            return data["data"]

    except Exception as e:
        return [{"value": f"Error: {e}", "created_at": ""}]

    return []

def fetch_latest_sensor(feed):
    samples = fetch_feed_samples(feed, 1)

    if samples and "error" not in samples[0]:
        item = samples[0]
        return {
            "value": item.get("value", "N/A"),
            "time": item.get("created_at", "N/A")
        }

    return {"value": "N/A", "time": "No data"}

def fetch_latest_json_sample():
    samples = fetch_feed_samples("json", 1)

    if not samples:
        return None

    item = samples[0]
    raw_value = item.get("value", "{}")

    try:
        if isinstance(raw_value, str):
            raw_value = json.loads(raw_value)

        return {
            "temperature": float(raw_value.get("temperature", 0)),
            "humidity": float(raw_value.get("humidity", 0)),
            "soil": float(raw_value.get("soil", 0)),
            "sensor_timestamp": item.get("created_at", ""),
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "source": "Render Server / Adafruit IO"
        }

    except Exception as e:
        print("JSON parse error:", e)
        return None

# =====================================================
# 3) FIREBASE SAVE
# =====================================================
def firebase_key_from_timestamp(timestamp):
    return (
        str(timestamp)
        .replace(":", "-")
        .replace(".", "-")
        .replace("/", "-")
        .replace("Z", "")
    )

def save_sensor_sample_to_firebase(sample):
    if not sample:
        return False, "No sample to save."

    if not sample.get("sensor_timestamp"):
        return False, "Sample has no timestamp."

    key = firebase_key_from_timestamp(sample["sensor_timestamp"])
    url = f"{FIREBASE_DB_URL}/sensor_history/{key}.json"

    try:
        check = requests.get(url, timeout=10)

        if check.status_code == 200 and check.json() is not None:
            return False, "Sample already exists in Firebase."

        save = requests.put(url, json=sample, timeout=10)
        save.raise_for_status()

        return True, "New sensor sample saved to Firebase."

    except Exception as e:
        return False, f"Firebase save error: {e}"

# =====================================================
# 4) HELPERS
# =====================================================
def format_time(raw_time):
    try:
        dt = datetime.fromisoformat(str(raw_time).replace("Z", "+00:00"))
        return dt.strftime("%d/%m/%Y %H:%M:%S")
    except:
        return raw_time

def to_float(value):
    try:
        return float(value)
    except:
        return None

def calculate_stats(feed, limit):
    samples = fetch_feed_samples(feed, limit)
    values = []

    for sample in samples:
        val = to_float(sample.get("value"))
        if val is not None:
            values.append(val)

    if not values:
        return {
            "avg": "N/A",
            "min": "N/A",
            "max": "N/A",
            "count": 0
        }

    return {
        "avg": round(sum(values) / len(values), 2),
        "min": round(min(values), 2),
        "max": round(max(values), 2),
        "count": len(values)
    }

# =====================================================
# 5) STYLE
# =====================================================
iot_style = widgets.HTML("""
<style>
.iot-screen {
    font-family: Arial, sans-serif;
}

.iot-shell {
    background: linear-gradient(135deg, #f8fafc 0%, #f0fdf4 100%);
    border-radius: 30px;
    padding: 34px;
    border: 1px solid #e5e7eb;
}

.iot-title {
    font-size: 36px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 6px;
}

.iot-subtitle {
    font-size: 15px;
    color: #64748b;
    margin-bottom: 28px;
}

.iot-card {
    background: white;
    border-radius: 26px;
    padding: 28px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 14px 34px rgba(15,23,42,0.07);
}

.sensor-card {
    background:
      radial-gradient(circle at top left, rgba(34,197,94,0.12), transparent 35%),
      linear-gradient(180deg,#ffffff,#f8fafc);
    border-radius: 24px;
    padding: 22px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 10px 24px rgba(15,23,42,0.06);
    min-height: 190px;
    cursor: pointer;
    transition: all 0.2s ease-in-out;
}

.sensor-card:hover {
    transform: translateY(-4px);
    border-color: #22c55e;
    box-shadow: 0 16px 28px rgba(34,197,94,0.16);
}

.sensor-icon {
    font-size: 34px;
    margin-bottom: 8px;
}

.sensor-label {
    font-size: 15px;
    font-weight: 800;
    color: #64748b;
}

.sensor-value {
    font-size: 36px;
    font-weight: 900;
    color: #14532d;
    margin-top: 8px;
}

.sensor-time {
    font-size: 12px;
    color: #94a3b8;
    margin-top: 8px;
}

.sensor-stats {
    font-size: 12px;
    color: #334155;
    background: #f8fafc;
    border-radius: 12px;
    padding: 8px;
    margin-top: 12px;
    line-height: 1.6;
}

.refresh-btn-iot {
    background: linear-gradient(180deg, #22c55e 0%, #166534 100%) !important;
    color: white !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: none !important;
}

.info-box {
    background: #ffffff;
    border-left: 5px solid #22c55e;
    border-radius: 16px;
    padding: 18px;
    font-size: 14px;
    color: #334155;
    line-height: 1.6;
}

.raw-box {
    background: #f8fafc;
    border-radius: 18px;
    padding: 18px;
    border: 1px solid #e5e7eb;
    margin-top: 18px;
    font-family: monospace;
    font-size: 13px;
    color: #111827;
    max-height: 320px;
    overflow-y: auto;
}

.raw-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 10px;
}
</style>
""")

# =====================================================
# 6) UI ELEMENTS
# =====================================================
status_html = widgets.HTML()
feed_details_html = widgets.HTML()
db_status_html = widgets.HTML()

temp_card_html = widgets.HTML()
hum_card_html = widgets.HTML()
soil_card_html = widgets.HTML()

history_slider = widgets.IntSlider(
    value=30,
    min=5,
    max=300,
    step=5,
    description="History:",
    layout=Layout(width="420px")
)

temp_btn = widgets.Button(
    description="View Temperature Feed",
    layout=Layout(width="100%", height="42px")
)
hum_btn = widgets.Button(
    description="View Humidity Feed",
    layout=Layout(width="100%", height="42px")
)
soil_btn = widgets.Button(
    description="View Soil Feed",
    layout=Layout(width="100%", height="42px")
)

for btn in [temp_btn, hum_btn, soil_btn]:
    btn.add_class("refresh-btn-iot")

btn_refresh_iot = widgets.Button(
    description="Refresh Sensor Data",
    icon="refresh",
    layout=Layout(width="220px", height="52px")
)
btn_refresh_iot.add_class("refresh-btn-iot")

header = widgets.HTML("""
<div>
  <div class="iot-title">IoT Sensor Data Sampling 🌱</div>
  <div class="iot-subtitle">
    Real sensor readings retrieved from the cloud, then saved into Firebase database.
  </div>
</div>
""")

# =====================================================
# 7) RENDER FUNCTIONS
# =====================================================
def make_sensor_card(icon, label, value, unit, time, stats):
    return f"""
    <div class="sensor-card">
        <div class="sensor-icon">{icon}</div>
        <div class="sensor-label">{label}</div>
        <div class="sensor-value">{value}<span style="font-size:20px;"> {unit}</span></div>
        <div class="sensor-time">Last update: {format_time(time)}</div>
        <div class="sensor-stats">
            Average: <b>{stats['avg']}</b> {unit}<br>
            Min: <b>{stats['min']}</b> {unit} | Max: <b>{stats['max']}</b> {unit}<br>
            Samples used: <b>{stats['count']}</b>
        </div>
    </div>
    """

def update_cards():
    selected_limit = history_slider.value

    temperature = fetch_latest_sensor("temperature")
    humidity = fetch_latest_sensor("humidity")
    soil = fetch_latest_sensor("soil")

    temp_stats = calculate_stats("temperature", selected_limit)
    hum_stats = calculate_stats("humidity", selected_limit)
    soil_stats = calculate_stats("soil", selected_limit)

    temp_card_html.value = make_sensor_card(
        "🌡️", "Temperature", temperature["value"], "°C", temperature["time"], temp_stats
    )

    hum_card_html.value = make_sensor_card(
        "💧", "Humidity", humidity["value"], "%", humidity["time"], hum_stats
    )

    soil_card_html.value = make_sensor_card(
        "🌱", "Soil Moisture", soil["value"], "%", soil["time"], soil_stats
    )

def render_feed_details(feed, label, unit):
    selected_limit = history_slider.value
    samples = fetch_feed_samples(feed, selected_limit)

    if not samples:
        feed_details_html.value = """
        <div class="raw-box">
            <div class="raw-title">No data found</div>
        </div>
        """
        return

    values = []
    rows = ""

    for sample in samples:
        raw_value = sample.get("value", "N/A")
        numeric = to_float(raw_value)

        if numeric is not None:
            values.append(numeric)

        rows += f"{raw_value} {unit} &nbsp;&nbsp; {format_time(sample.get('created_at',''))}<br>"

    avg_val = round(sum(values) / len(values), 2) if values else "N/A"
    min_val = round(min(values), 2) if values else "N/A"
    max_val = round(max(values), 2) if values else "N/A"

    feed_details_html.value = f"""
    <div class="raw-box">
        <div class="raw-title">{label} Feed Details</div>
        Feed: <b>{feed}</b> | Samples requested: <b>{selected_limit}</b> | Samples received: <b>{len(samples)}</b><br>
        Average: <b>{avg_val}</b> {unit} |
        Min: <b>{min_val}</b> {unit} |
        Max: <b>{max_val}</b> {unit}
        <br><br>
        {rows}
    </div>
    """

def refresh_iot_data(_=None):
    update_cards()

    latest_sample = fetch_latest_json_sample()
    saved, message = save_sensor_sample_to_firebase(latest_sample)

    if saved:
        db_color = "#16a34a"
        db_icon = "✅"
    else:
        db_color = "#64748b"
        db_icon = "ℹ️"

    render_feed_details("temperature", "Temperature", "°C")

# =====================================================
# 8) CLICK EVENTS
# =====================================================
temp_btn.on_click(lambda _: render_feed_details("temperature", "Temperature", "°C"))
hum_btn.on_click(lambda _: render_feed_details("humidity", "Humidity", "%"))
soil_btn.on_click(lambda _: render_feed_details("soil", "Soil Moisture", "%"))
btn_refresh_iot.on_click(refresh_iot_data)

def on_history_change(change):
    if change["name"] == "value":
        refresh_iot_data()

history_slider.observe(on_history_change, names="value")

# =====================================================
# 9) FINAL SCREEN LAYOUT
# =====================================================
temp_card = widgets.VBox([temp_card_html, temp_btn], layout=Layout(width="32%"))
hum_card = widgets.VBox([hum_card_html, hum_btn], layout=Layout(width="32%"))
soil_card = widgets.VBox([soil_card_html, soil_btn], layout=Layout(width="32%"))

cards_row = widgets.HBox(
    [temp_card, hum_card, soil_card],
    layout=Layout(gap="22px", width="100%")
)

controls_row = widgets.HBox(
    [history_slider, btn_refresh_iot],
    layout=Layout(
        justify_content="space-between",
        align_items="center",
        margin="24px 0 0 0"
    )
)

main_card = widgets.VBox(
    [
        cards_row,
        controls_row,
        widgets.HTML("<div style='height:18px;'></div>"),
        status_html,
        db_status_html,
        feed_details_html
    ]
)
main_card.add_class("iot-card")

iot_screen = widgets.VBox(
    [
        iot_style,
        header,
        main_card
    ],
    layout=Layout(width="98%", margin="10px auto")
)

iot_screen.add_class("iot-shell")
iot_screen.add_class("iot-screen")

refresh_iot_data()



## 📊 Dashboard Screen

In [ ]:
# =====================================================
# SCREEN 4: PLANT HEALTH DASHBOARD FROM FIREBASE DB
# Big Data + MapReduce + User Selected Graph + Dates + Agent Logic
# =====================================================

import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display

# =====================================================
# 1) FIREBASE CONFIG
# =====================================================
FIREBASE_DB_URL = "https://smart-tomato-care-default-rtdb.firebaseio.com"
SENSOR_HISTORY_PATH = f"{FIREBASE_DB_URL}/sensor_history.json"

# =====================================================
# 2) LOAD DATA FROM FIREBASE
# =====================================================
def load_sensor_history(limit=300):
    try:
        response = requests.get(SENSOR_HISTORY_PATH, timeout=20)
        data = response.json()

        if not data:
            return []

        samples = list(data.values())
        samples.sort(key=lambda x: x.get("sensor_timestamp", ""))

        return samples[-limit:]

    except Exception as e:
        print("Firebase load error:", e)
        return []


def to_float(value):
    try:
        return float(value)
    except:
        return None


# =====================================================
# 3) MAP REDUCE FUNCTIONS
# =====================================================
def map_reduce_stats(samples, field):
    mapped_values = []

    # MAP STEP
    for sample in samples:
        value = to_float(sample.get(field))

        if value is not None:
            mapped_values.append(value)

    # REDUCE STEP
    if not mapped_values:
        return {
            "latest": "N/A",
            "avg": "N/A",
            "min": "N/A",
            "max": "N/A",
            "count": 0
        }

    return {
        "latest": round(mapped_values[-1], 2),
        "avg": round(sum(mapped_values) / len(mapped_values), 2),
        "min": round(min(mapped_values), 2),
        "max": round(max(mapped_values), 2),
        "count": len(mapped_values)
    }


# =====================================================
# 4) RISK CALCULATION
# =====================================================
def calculate_risk(temp, humidity, soil):
    risk = 0
    reasons = []

    if temp != "N/A" and temp is not None:
        if temp >= 35:
            risk += 35
            reasons.append("High temperature detected.")
        elif temp >= 32:
            risk += 20
            reasons.append("Temperature is slightly high.")

    if humidity != "N/A" and humidity is not None:
        if humidity < 35:
            risk += 25
            reasons.append("Humidity is low.")
        elif humidity > 85:
            risk += 20
            reasons.append("Humidity is too high.")

    if soil != "N/A" and soil is not None:
        if soil < 30:
            risk += 35
            reasons.append("Soil moisture is very low.")
        elif soil < 45:
            risk += 20
            reasons.append("Soil moisture is dropping.")

    risk = min(risk, 100)

    if risk < 30:
        return risk, "Healthy", "🟢", "#16a34a", reasons or ["Sensor readings are within a normal range."]
    elif risk < 60:
        return risk, "Needs Attention", "🟠", "#f97316", reasons
    else:
        return risk, "High Risk", "🔴", "#dc2626", reasons


# =====================================================
# 5) AGENT LOGIC
# =====================================================
def plant_agent(df):
    """
    Agent Logic:
    The agent analyzes recent historical sensor data,
    detects trends, and gives smart recommendations.
    """

    recommendations = []

    if df.empty or len(df) < 5:
        return [
            "📌 Not enough historical data yet. Continue collecting sensor readings."
        ]

    recent_df = df.tail(10)

    temp_avg = recent_df["temperature"].mean()
    humidity_avg = recent_df["humidity"].mean()
    soil_avg = recent_df["soil"].mean()
    risk_avg = recent_df["risk"].mean()

    temp_first = recent_df["temperature"].iloc[0]
    temp_last = recent_df["temperature"].iloc[-1]

    soil_first = recent_df["soil"].iloc[0]
    soil_last = recent_df["soil"].iloc[-1]

    humidity_first = recent_df["humidity"].iloc[0]
    humidity_last = recent_df["humidity"].iloc[-1]

    risk_first = recent_df["risk"].iloc[0]
    risk_last = recent_df["risk"].iloc[-1]

    # Temperature analysis
    if temp_avg >= 35:
        recommendations.append(
            "🌡️ Temperature stayed very high in recent readings. Move the plant to a cooler place or reduce direct sunlight."
        )
    elif temp_avg >= 32:
        recommendations.append(
            "🌡️ Temperature is slightly high. Keep monitoring and avoid long exposure to heat."
        )

    if temp_last > temp_first + 3:
        recommendations.append(
            "📈 Temperature is increasing over time. This may stress the plant if it continues."
        )

    # Soil moisture analysis
    if soil_avg < 30:
        recommendations.append(
            "🌱 Soil moisture is very low. Water the plant as soon as possible."
        )
    elif soil_avg < 45:
        recommendations.append(
            "💧 Soil moisture is dropping. Check if the plant needs watering."
        )

    if soil_last < soil_first - 10:
        recommendations.append(
            "📉 Soil moisture decreased clearly in the last readings. Increase irrigation frequency."
        )

    # Humidity analysis
    if humidity_avg < 35:
        recommendations.append(
            "🌬️ Humidity is low. Try to increase air humidity around the plant."
        )
    elif humidity_avg > 85:
        recommendations.append(
            "💦 Humidity is too high. Improve ventilation to avoid plant disease."
        )

    if humidity_last < humidity_first - 10:
        recommendations.append(
            "📉 Humidity is decreasing. The environment may become too dry."
        )

    # Risk analysis
    if risk_avg >= 60:
        recommendations.append(
            "🔴 The average risk is high. Immediate care is recommended."
        )
    elif risk_last > risk_first + 20:
        recommendations.append(
            "⚠️ Risk score increased recently. Check temperature, humidity, and soil moisture."
        )

    if len(recommendations) == 0:
        recommendations.append(
            "✅ The plant conditions look stable. Continue regular monitoring."
        )

    return recommendations


# =====================================================
# 6) STYLE
# =====================================================
dashboard_style = widgets.HTML("""
<style>
.dashboard-shell {
    background: linear-gradient(135deg, #f8fafc 0%, #f0fdf4 100%);
    border-radius: 30px;
    padding: 34px;
    border: 1px solid #e5e7eb;
    font-family: Arial, sans-serif;
}

.dash-title {
    font-size: 36px;
    font-weight: 900;
    color: #14532d;
    margin-bottom: 6px;
}

.dash-subtitle {
    font-size: 15px;
    color: #64748b;
    margin-bottom: 28px;
}

.dash-card {
    background: white;
    border-radius: 26px;
    padding: 26px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 14px 34px rgba(15,23,42,0.07);
}

.metric-card {
    background: linear-gradient(180deg,#ffffff,#f8fafc);
    border-radius: 22px;
    padding: 22px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 8px 20px rgba(15,23,42,0.05);
}

.metric-title {
    font-size: 14px;
    color: #64748b;
    font-weight: 800;
}

.metric-value {
    font-size: 34px;
    color: #14532d;
    font-weight: 900;
    margin-top: 8px;
}

.metric-small {
    font-size: 12px;
    color: #64748b;
    margin-top: 8px;
    line-height: 1.6;
}

.refresh-dashboard-btn {
    background: linear-gradient(180deg, #22c55e 0%, #166534 100%) !important;
    color: white !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    border: none !important;
}

.agent-box {
    background:#f0fdf4;
    border-left:5px solid #22c55e;
    border-radius:14px;
    padding:16px;
    color:#14532d;
    font-weight:700;
    line-height:1.8;
}
</style>
""")

# =====================================================
# 7) UI ELEMENTS
# =====================================================
header = widgets.HTML("""
<div class="dash-title">Plant Health Dashboard 🍅</div>
<div class="dash-subtitle">
Dashboard based on Firebase sensor data with Big Data, MapReduce, and Agent recommendations.
</div>
""")

summary_html = widgets.HTML()
metrics_html = widgets.HTML()
recommendations_html = widgets.HTML()
mapreduce_html = widgets.HTML()

chart_output = widgets.Output()
table_output = widgets.Output()

samples_slider = widgets.IntSlider(
    value=100,
    min=5,
    max=500,
    step=5,
    description="History:",
    layout=Layout(width="360px")
)

progress_slider = widgets.IntSlider(
    value=30,
    min=5,
    max=200,
    step=5,
    description="Progress:",
    layout=Layout(width="360px")
)

metric_dropdown = widgets.Dropdown(
    options=[
        ("Temperature", "temperature"),
        ("Humidity", "humidity"),
        ("Soil Moisture", "soil"),
        ("Risk Score", "risk")
    ],
    value="temperature",
    description="Graph:",
    layout=Layout(width="300px")
)

date_mode_dropdown = widgets.Dropdown(
    options=[
        ("Date + Time", "datetime"),
        ("Days Only", "day")
    ],
    value="datetime",
    description="X Axis:",
    layout=Layout(width="300px")
)

btn_refresh_dashboard = widgets.Button(
    description="Refresh Dashboard",
    icon="refresh",
    layout=Layout(width="220px", height="50px")
)

btn_refresh_dashboard.add_class("refresh-dashboard-btn")


# =====================================================
# 8) RENDER DASHBOARD
# =====================================================
def render_dashboard(_=None):
    samples = load_sensor_history(samples_slider.value)

    if not samples:
        summary_html.value = """
        <div class="dash-card">
            <h3 style="color:#dc2626;">No data found in Firebase.</h3>
            <p>Please refresh the IoT screen first to save sensor data into the database.</p>
        </div>
        """
        metrics_html.value = ""
        recommendations_html.value = ""
        mapreduce_html.value = ""

        with chart_output:
            chart_output.clear_output()

        return

    # Big Data + MapReduce for all main sensors
    temp_stats = map_reduce_stats(samples, "temperature")
    hum_stats = map_reduce_stats(samples, "humidity")
    soil_stats = map_reduce_stats(samples, "soil")

    risk, status, icon, color, reasons = calculate_risk(
        temp_stats["latest"],
        hum_stats["latest"],
        soil_stats["latest"]
    )

    summary_html.value = f"""
    <div class="dash-card" style="border-left:8px solid {color};">
        <div style="font-size:28px;font-weight:900;color:{color};">
            {icon} Plant Status: {status}
        </div>

        <div style="font-size:44px;font-weight:900;color:#111827;margin-top:10px;">
            Risk Score: {risk}%
        </div>

        <div style="font-size:14px;color:#64748b;margin-top:8px;">
            Analysis is based on Big Data records saved in Firebase.
        </div>
    </div>
    """

    metrics_html.value = f"""
    <div style="display:flex;gap:18px;margin-top:22px;flex-wrap:wrap;">
        <div class="metric-card" style="flex:1;min-width:220px;">
            <div style="font-size:30px;">🌡️</div>
            <div class="metric-title">Temperature</div>
            <div class="metric-value">{temp_stats['latest']} °C</div>
            <div class="metric-small">
                Average: {temp_stats['avg']} °C<br>
                Min: {temp_stats['min']} | Max: {temp_stats['max']}<br>
                Records: {temp_stats['count']}
            </div>
        </div>

        <div class="metric-card" style="flex:1;min-width:220px;">
            <div style="font-size:30px;">💧</div>
            <div class="metric-title">Humidity</div>
            <div class="metric-value">{hum_stats['latest']} %</div>
            <div class="metric-small">
                Average: {hum_stats['avg']} %<br>
                Min: {hum_stats['min']} | Max: {hum_stats['max']}<br>
                Records: {hum_stats['count']}
            </div>
        </div>

        <div class="metric-card" style="flex:1;min-width:220px;">
            <div style="font-size:30px;">🌱</div>
            <div class="metric-title">Soil Moisture</div>
            <div class="metric-value">{soil_stats['latest']} %</div>
            <div class="metric-small">
                Average: {soil_stats['avg']} %<br>
                Min: {soil_stats['min']} | Max: {soil_stats['max']}<br>
                Records: {soil_stats['count']}
            </div>
        </div>
    </div>
    """

    # ==============================
    # DataFrame
    # ==============================
    df = pd.DataFrame(samples)

    for col in ["temperature", "humidity", "soil"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["risk"] = df.apply(
        lambda row: calculate_risk(
            row["temperature"],
            row["humidity"],
            row["soil"]
        )[0],
        axis=1
    )

    df["datetime_obj"] = pd.to_datetime(df["sensor_timestamp"], errors="coerce")
    df["date_time_label"] = df["datetime_obj"].dt.strftime("%d/%m %H:%M")
    df["day_label"] = df["datetime_obj"].dt.strftime("%d/%m/%Y")

    df["date_time_label"] = df["date_time_label"].fillna(df["sensor_timestamp"].astype(str))
    df["day_label"] = df["day_label"].fillna(df["sensor_timestamp"].astype(str))

    # ==============================
    # Agent Recommendations
    # ==============================
    agent_recommendations = plant_agent(df)

    agent_items = "".join([f"<li>{rec}</li>" for rec in agent_recommendations])

    recommendations_html.value = f"""
    <div class="dash-card" style="margin-top:22px;">
        <div style="font-size:22px;font-weight:900;color:#14532d;margin-bottom:10px;">
            Smart Plant Agent Recommendations 🤖
        </div>

        <div style="font-size:14px;color:#64748b;margin-bottom:12px;">
            The agent analyzes recent sensor trends and suggests actions for the user.
        </div>

        <div class="agent-box">
            <ul style="margin:0;padding-left:20px;">
                {agent_items}
            </ul>
        </div>
    </div>
    """

    # ==============================
    # User Selected Progress Graph
    # ==============================
    selected_metric = metric_dropdown.value
    progress_count = progress_slider.value
    x_axis_mode = date_mode_dropdown.value

    progress_df = df.tail(progress_count)

    selected_samples = progress_df.to_dict("records")
    selected_stats = map_reduce_stats(selected_samples, selected_metric)

    if x_axis_mode == "day":
        x_values = progress_df["day_label"]
        x_title = "Date"
    else:
        x_values = progress_df["date_time_label"]
        x_title = "Date and Time"

    mapreduce_html.value = f"""
    <div class="dash-card" style="margin-top:22px;">
        <div style="font-size:22px;font-weight:900;color:#14532d;">
    Plant Health Insights 🍅
</div>

        <div style="font-size:14px;color:#334155;line-height:1.8;margin-top:10px;">
            <b>Selected Metric:</b> {selected_metric}<br>
<b>Average Value:</b> {selected_stats['avg']}<br>
<b>Lowest Value:</b> {selected_stats['min']}<br>
<b>Highest Value:</b> {selected_stats['max']}<br>
<b>Current Value:</b> {selected_stats['latest']}<br>
<b>Measurements Analyzed:</b> {selected_stats['count']}
        </div>
    </div>
    """

    with chart_output:
        chart_output.clear_output()

        plt.figure(figsize=(12, 4))

        plt.plot(
            x_values,
            progress_df[selected_metric],
            marker="o",
            label=selected_metric
        )

        plt.title(f"Plant Progress By {x_title} - {selected_metric}")
        plt.xlabel(x_title)
        plt.ylabel("Value")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()



# =====================================================
# 9) EVENTS
# =====================================================
btn_refresh_dashboard.on_click(render_dashboard)

metric_dropdown.observe(render_dashboard, names="value")
progress_slider.observe(render_dashboard, names="value")
samples_slider.observe(render_dashboard, names="value")
date_mode_dropdown.observe(render_dashboard, names="value")


# =====================================================
# 10) FINAL LAYOUT
# =====================================================
dashboard_screen = widgets.VBox([
    dashboard_style,
    header,

    widgets.HBox(
        [
            samples_slider,
            progress_slider,
            metric_dropdown,
            date_mode_dropdown,
            btn_refresh_dashboard
        ],
        layout=Layout(
            justify_content="space-between",
            align_items="center",
            flex_flow="row wrap",
            gap="12px"
        )
    ),

    widgets.HTML("<br>"),

    summary_html,
    metrics_html,
    recommendations_html,
    mapreduce_html,

    widgets.HTML("<br>"),
    widgets.VBox([chart_output], layout=Layout(margin="10px 0")),

], layout=Layout(width="98%", margin="10px auto"))

dashboard_screen.add_class("dashboard-shell")

render_dashboard()



## 🎮 Tomato Catcher Game

In [ ]:
from IPython.display import display, HTML

tomato_game_html = """
<div id="tomato-game-wrapper">
    <style>
        #tomato-game-wrapper {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            width: 92%;
            margin: 20px auto;
        }

        .game-header {
            background: linear-gradient(135deg,#ffffff,#f0fdf4);
            border: 1px solid #dceee5;
            border-radius: 26px;
            padding: 28px;
            box-shadow: 0 12px 34px rgba(15,118,59,0.08);
            margin-bottom: 18px;
        }

        .game-title {
            font-size: 38px;
            font-weight: 900;
            color: #14532d;
        }

        .game-subtitle {
            font-size: 17px;
            color: #64748b;
            margin-top: 8px;
        }

        .score-board {
            display: flex;
            justify-content: center;
            gap: 18px;
            margin-bottom: 16px;
            flex-wrap: wrap;
        }

        .score-card {
            background: white;
            border: 1px solid #bbf7d0;
            border-radius: 18px;
            padding: 12px 22px;
            color: #166534;
            font-size: 18px;
            font-weight: 900;
            box-shadow: 0 8px 20px rgba(15,118,59,0.08);
        }

        #game-area {
            position: relative;
            width: 100%;
            height: 440px;
            background: linear-gradient(180deg,#ecfdf5,#ffffff);
            border: 2px solid #bbf7d0;
            border-radius: 28px;
            overflow: hidden;
            box-shadow: 0 15px 40px rgba(15,118,59,0.10);
        }

        #falling-tomato {
            position: absolute;
            font-size: 54px;
            top: -70px;
            left: 50%;
            transform: translateX(-50%);
            will-change: top, left;
            user-select: none;
            z-index: 5;
        }

        #basket {
            position: absolute;
            font-size: 76px;
            bottom: 28px;
            left: 50%;
            transform: translateX(-50%);
            transition: left 0.16s ease-out;
            will-change: left;
            user-select: none;
            z-index: 4;
        }

        .game-pill {
            position: absolute;
            left: 20px;
            background: white;
            border-radius: 999px;
            padding: 8px 16px;
            font-weight: 900;
            z-index: 3;
        }

        .pill-good {
            top: 15px;
            color: #14532d;
            border: 1px solid #bbf7d0;
        }

        .pill-bad {
            top: 60px;
            color: #7c2d12;
            border: 1px solid #fed7aa;
        }

        #game-message {
            text-align: center;
            font-size: 20px;
            font-weight: 900;
            color: #14532d;
            margin-top: 18px;
            min-height: 30px;
        }

        .controls {
            display: flex;
            justify-content: center;
            gap: 12px;
            margin-top: 16px;
            flex-wrap: wrap;
        }

        .game-btn {
            border: none;
            border-radius: 16px;
            padding: 12px 22px;
            font-size: 16px;
            font-weight: 900;
            cursor: pointer;
            color: white;
            box-shadow: 0 10px 24px rgba(22,101,52,0.22);
        }

        .green-btn {
            background: linear-gradient(180deg,#22c55e,#166534);
        }

        .orange-btn {
            background: linear-gradient(180deg,#fb923c,#c2410c);
        }

        .game-btn:hover {
            filter: brightness(0.95);
        }

        .hint {
            text-align: center;
            color: #64748b;
            margin-top: 10px;
            font-size: 14px;
        }

        #game-over-overlay {
            position: absolute;
            inset: 0;
            background: rgba(20, 83, 45, 0.78);
            display: none;
            align-items: center;
            justify-content: center;
            z-index: 20;
        }

        .game-over-box {
            background: white;
            border-radius: 28px;
            padding: 36px 48px;
            text-align: center;
            box-shadow: 0 20px 50px rgba(0,0,0,0.25);
            border: 3px solid #ef4444;
        }

        .game-over-title {
            font-size: 52px;
            font-weight: 900;
            color: #dc2626;
            letter-spacing: 2px;
        }

        .game-over-subtitle {
            font-size: 20px;
            font-weight: 700;
            color: #14532d;
            margin-top: 10px;
            margin-bottom: 24px;
        }

        .game-over-score {
            font-size: 18px;
            font-weight: 900;
            color: #166534;
            margin-bottom: 24px;
        }

        .game-over-btn {
            border: none;
            border-radius: 16px;
            padding: 13px 26px;
            background: linear-gradient(180deg,#22c55e,#166534);
            color: white;
            font-size: 17px;
            font-weight: 900;
            cursor: pointer;
            box-shadow: 0 10px 24px rgba(22,101,52,0.28);
        }

        .game-over-btn:hover {
            filter: brightness(0.95);
        }
    </style>

    <div class="game-header">
        <div class="game-title">Tomato Catcher 🍅</div>
        <div class="game-subtitle">
            תפוס רק את העגבניות הבריאות. אם תפסת עגבנייה פגומה — המשחק נגמר!
        </div>
    </div>

    <div class="score-board">
        <div class="score-card">Score: <span id="score">0</span></div>
        <div class="score-card">Level: <span id="level">1</span></div>
        <div class="score-card">Speed: <span id="speed">Normal</span></div>
    </div>

    <div id="game-area">
        <div id="game-over-overlay">
            <div class="game-over-box">
                <div class="game-over-title">GAME OVER</div>
                <div class="game-over-subtitle">You caught a damaged tomato 🍅</div>
                <div class="game-over-score">Final Score: <span id="final-score">0</span></div>
                <button class="game-over-btn" onclick="restartTomatoGame()">Play Again</button>
            </div>
        </div>

        <div class="game-pill pill-good">Healthy 🍅 = +10 points</div>
        <div class="game-pill pill-bad">Rotten 🦠🍅 = Game Over</div>

        <div id="falling-tomato">🍅</div>
        <div id="basket">🧺</div>
    </div>

    <div id="game-message">Press Start Game to begin.</div>

    <div class="controls">
        <button class="game-btn green-btn" onclick="moveBasketLeft()">⬅️ Left</button>
        <button class="game-btn green-btn" onclick="startTomatoGame()">Start Game 🚀</button>
        <button class="game-btn orange-btn" onclick="restartTomatoGame()">Restart 🔄</button>
        <button class="game-btn green-btn" onclick="moveBasketRight()">Right ➡️</button>
    </div>

    <div class="hint">
        You can also use keyboard arrows ⬅️ ➡️ to move the basket. Sound starts after pressing Start.
    </div>

    <script>
        let gameRunning = false;
        let score = 0;
        let level = 1;

        let basketX = 50;
        let tomatoX = 50;
        let tomatoY = -70;
        let tomatoType = "good";

        let fallSpeed = 2.2;
        let animationId = null;

        const tomato = document.getElementById("falling-tomato");
        const basket = document.getElementById("basket");
        const scoreDisplay = document.getElementById("score");
        const levelDisplay = document.getElementById("level");
        const speedDisplay = document.getElementById("speed");
        const message = document.getElementById("game-message");
        const gameArea = document.getElementById("game-area");
        const gameOverOverlay = document.getElementById("game-over-overlay");
        const finalScore = document.getElementById("final-score");

        // -----------------------------
        // Sound effects using Web Audio
        // -----------------------------
        let audioContext = null;

        function getAudioContext() {
            if (!audioContext) {
                audioContext = new (window.AudioContext || window.webkitAudioContext)();
            }

            if (audioContext.state === "suspended") {
                audioContext.resume();
            }

            return audioContext;
        }

        function playTone(frequency, duration, type = "sine", volume = 0.15) {
            const ctx = getAudioContext();

            const oscillator = ctx.createOscillator();
            const gainNode = ctx.createGain();

            oscillator.type = type;
            oscillator.frequency.setValueAtTime(frequency, ctx.currentTime);

            gainNode.gain.setValueAtTime(volume, ctx.currentTime);
            gainNode.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + duration);

            oscillator.connect(gainNode);
            gainNode.connect(ctx.destination);

            oscillator.start();
            oscillator.stop(ctx.currentTime + duration);
        }

        function playStartSound() {
            playTone(500, 0.12, "sine", 0.14);
            setTimeout(() => playTone(700, 0.12, "sine", 0.14), 100);
        }

        function playCatchSound() {
            playTone(650, 0.12, "sine", 0.18);
            setTimeout(() => playTone(850, 0.10, "sine", 0.14), 90);
        }

        function playMissSound() {
            playTone(260, 0.12, "triangle", 0.12);
        }

        function playGameOverSound() {
            playTone(300, 0.20, "sawtooth", 0.16);
            setTimeout(() => playTone(220, 0.25, "sawtooth", 0.14), 160);
            setTimeout(() => playTone(150, 0.35, "sawtooth", 0.12), 340);
        }

        function updateBasket() {
            basket.style.left = basketX + "%";
        }

        function updateTomato() {
            tomato.style.left = tomatoX + "%";
            tomato.style.top = tomatoY + "px";
        }

        function moveBasketLeft() {
            if (!gameRunning && score === 0) {
                message.innerHTML = "Press Start Game first 🍅";
            }

            basketX -= 10;

            if (basketX < 10) {
                basketX = 10;
            }

            updateBasket();
        }

        function moveBasketRight() {
            if (!gameRunning && score === 0) {
                message.innerHTML = "Press Start Game first 🍅";
            }

            basketX += 10;

            if (basketX > 90) {
                basketX = 90;
            }

            updateBasket();
        }

        function randomTomato() {
            tomatoX = [10, 20, 30, 40, 50, 60, 70, 80, 90][Math.floor(Math.random() * 9)];
            tomatoY = -70;

            const randomValue = Math.random();

            if (randomValue < 0.75) {
                tomatoType = "good";
                tomato.innerHTML = "🍅";
            } else {
                tomatoType = "bad";
                tomato.innerHTML = "🦠🍅";
            }

            updateTomato();
        }

        function updateScore(points) {
            score += points;
            scoreDisplay.innerHTML = score;

            level = Math.floor(score / 50) + 1;
            levelDisplay.innerHTML = level;

            fallSpeed = 2.2 + (level - 1) * 0.35;

            if (level <= 2) {
                speedDisplay.innerHTML = "Normal";
            } else if (level <= 4) {
                speedDisplay.innerHTML = "Fast";
            } else {
                speedDisplay.innerHTML = "Expert";
            }
        }

        function checkCollision() {
            const basketRect = basket.getBoundingClientRect();
            const tomatoRect = tomato.getBoundingClientRect();

            const horizontalOverlap =
                tomatoRect.left < basketRect.right &&
                tomatoRect.right > basketRect.left;

            const verticalOverlap =
                tomatoRect.bottom > basketRect.top &&
                tomatoRect.top < basketRect.bottom;

            return horizontalOverlap && verticalOverlap;
        }

        function showGameOver() {
            gameRunning = false;

            if (animationId) {
                cancelAnimationFrame(animationId);
            }

            playGameOverSound();

            finalScore.innerHTML = score;
            gameOverOverlay.style.display = "flex";
            message.innerHTML = "❌ Game Over! You caught a damaged tomato.";
        }

        function gameLoop() {
            if (!gameRunning) {
                return;
            }

            tomatoY += fallSpeed;
            updateTomato();

            if (checkCollision()) {
                if (tomatoType === "good") {
                    updateScore(10);
                    playCatchSound();
                    message.innerHTML = "✅ Great! Healthy tomato caught.";
                    randomTomato();
                } else {
                    showGameOver();
                    return;
                }
            }

            const areaHeight = gameArea.clientHeight;

            if (tomatoY > areaHeight + 40) {
                playMissSound();
                message.innerHTML = "🍃 Missed! Keep going.";
                randomTomato();
            }

            animationId = requestAnimationFrame(gameLoop);
        }

        function startTomatoGame() {
            if (gameRunning) {
                return;
            }

            gameOverOverlay.style.display = "none";
            gameRunning = true;
            message.innerHTML = "Game started! Catch only healthy tomatoes 🍅";

            playStartSound();

            randomTomato();
            animationId = requestAnimationFrame(gameLoop);
        }

        function restartTomatoGame() {
            gameRunning = false;

            if (animationId) {
                cancelAnimationFrame(animationId);
            }

            gameOverOverlay.style.display = "none";

            score = 0;
            level = 1;
            fallSpeed = 2.2;
            basketX = 50;
            tomatoX = 50;
            tomatoY = -70;
            tomatoType = "good";

            scoreDisplay.innerHTML = score;
            levelDisplay.innerHTML = level;
            speedDisplay.innerHTML = "Normal";
            tomato.innerHTML = "🍅";

            updateBasket();
            updateTomato();

            message.innerHTML = "Press Start Game to begin.";
        }

        document.addEventListener("keydown", function(event) {
            if (event.key === "ArrowLeft") {
                moveBasketLeft();
            }

            if (event.key === "ArrowRight") {
                moveBasketRight();
            }
        });

        updateBasket();
        updateTomato();
    </script>
</div>
"""


In [ ]:
# =====================================================
# CREATE TOMATO CATCHER TAB SCREEN
# =====================================================

import ipywidgets as widgets
from IPython.display import display, HTML

game_output = widgets.Output()

with game_output:
    display(HTML(tomato_game_html))

game_screen = widgets.VBox([
    game_output
])

## 🚀 Main Application

In [ ]:
# =====================================================
# MAIN APP NAVIGATION - ALL SCREENS TOGETHER
# =====================================================

import ipywidgets as widgets
from IPython.display import display

app_tabs = widgets.Tab()

app_tabs.children = [
    screen,            # Plant Scanner
    iot_screen,        # IoT Sensor Data
    dashboard_screen,  # Dashboard
    search_screen,     # Garden Search
    game_screen        # Tomato Catcher Game
]

app_tabs.set_title(0, "🍅 Plant Scanner")
app_tabs.set_title(1, "🌱 IoT Data")
app_tabs.set_title(2, "📊 Dashboard")
app_tabs.set_title(3, "🔍 Garden Search")
app_tabs.set_title(4, "🎮 Tomato Catcher")

app_header = widgets.HTML("""
<div style="
    background:linear-gradient(135deg,#f0fdf4,#ffffff);
    border:1px solid #e5e7eb;
    border-radius:24px;
    padding:24px;
    margin-bottom:18px;
    font-family:Arial;">

    <div style="font-size:34px;font-weight:900;color:#14532d;">
        Smart Tomato Care System 🍅
    </div>

    <div style="font-size:15px;color:#64748b;margin-top:6px;">
        Comprehensive monitoring, diagnosis, research search, and gamified learning tools.
    </div>
</div>
""")

main_app = widgets.VBox([
    app_header,
    app_tabs,
    widgets.HTML("<div style='height:20px;'></div>"),
    floating_chat
])

display(main_app)